In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [3]:
# EXPERIMENTAL V14 (RETURN CHALLENGER) -- paste this ENTIRE file into ONE Kaggle/Jupyter code cell.
# Development-only feature/model selection; previously inspected tests remain exploratory.
# Default HISTORY_MODE='daily': only earlier completed dates update test history.
# Optional 'frozen': all test-period outcomes excluded; model/policy stay fixed in both.
# Current-race odds, times, running positions and payouts are NEVER predictors.
# Real ROI requires the supplied runner workbook AND the full audited PLACE dividend CSV.
# A return above 1.000 is an objective, NOT a promised or fabricated result.
import os
for thread_variable in ['OMP_NUM_THREADS', 'MKL_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'NUMEXPR_NUM_THREADS']:
    os.environ.setdefault(thread_variable, '1')
import re
import json
import hashlib
import itertools
import time
import contextlib
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.linear_model import LogisticRegression
try:
    from lightgbm import LGBMClassifier, LGBMRegressor
    HAS_LIGHTGBM = True
except ImportError:
    HAS_LIGHTGBM = False
try:
    from catboost import CatBoostClassifier, CatBoostRegressor
    HAS_CATBOOST = True
except ImportError:
    HAS_CATBOOST = False
try:
    from IPython.display import display
except ImportError:
    display = print

DATA_PATH = '/kaggle/input/datasets/miwawa/train-2014-to-2022/train_2014_to_2022.xlsx'
ODDS_PATH = '/kaggle/input/datasets/miwawa/all-odds-1/hkjc_Complete_all_dividends_20140914_20220930.csv'
OUTPUT_DIR = '/kaggle/working/roi_research_v14_value' if Path('/kaggle').exists() else str(Path.cwd() / 'outputs' / 'roi_research_v14')
TRAIN_TEST_PERIODS = [
    dict(name='period_1', train_start='2014-09-14', train_end='2018-09-30', test_start='2018-10-01', test_end='2019-12-31'),
    dict(name='period_2', train_start='2017-09-14', train_end='2021-09-14', test_start='2021-09-15', test_end='2022-09-28'),
]
# Freeze these before evaluating the final test. Do not retune after reading it.
# CatBoost is optional; the notebook still runs if not installed.
# Retain ALL V9 profiles as controls; test one conservative compact V10 profile.
# Fixed development candidates: hold the learner settings equal to isolate V8
# feature effects versus the V7 comparator. Do not change these after test evaluation.
# Fixed before discovery and never chosen using previously seen final-test outcomes.
# V7 and V8 context baselines isolate the effect of the new V9 features.
MODEL_PROFILES = {
    'v7_context': {'engine': 'histgb', 'max_leaf_nodes': 7, 'feature_set': 'v7'},
    'v8_context': {'engine': 'histgb', 'max_leaf_nodes': 7, 'feature_set': 'v8'},
    'v9_tactical': {'engine': 'histgb', 'max_leaf_nodes': 7, 'feature_set': 'v9_lite'},
}
if HAS_CATBOOST:
    MODEL_PROFILES['v9_catboost'] = {'engine': 'catboost', 'depth': 5,
                                     'iterations': 240, 'feature_set': 'v9_lite'}
if HAS_LIGHTGBM:
    MODEL_PROFILES['v9_lightgbm'] = {'engine': 'lightgbm', 'num_leaves': 10,
                                     'iterations': 180, 'feature_set': 'v9_lite'}
# V10 compact profile: strong shrinkage + shorter history-decay, fewer features.
# Fixed prior to all discovery/confirmation comparisons.
MODEL_PROFILES['v10_compact'] = {'engine': 'histgb', 'max_leaf_nodes': 5,
                                 'feature_set': 'v10_compact', 'half_life_days': 365}
# The old ensemble is deliberately held fixed for fair V9-vs-V10 comparison:
# adding V10 to the ensemble would otherwise quietly change V9 baseline.
V9_ENSEMBLE_MEMBERS = [m for m in MODEL_PROFILES if m != 'v10_compact']
# One predeclared challenger: blend older stable ensemble and recent compact model.
# Fixed weights and model architecture do not use any previous final-test ROI.
V11_LEGACY_WEIGHT = 0.65
V11_COMPACT_WEIGHT = 1.0 - V11_LEGACY_WEIGHT
V11_HYBRID_NAME = 'v11_hybrid'
# Three predeclared venues; evaluate scoped strategies only on development.
# Avoid an unconstrained search across dozens of unstable subgroups.
V12_SCOPED_MODELS = ('ensemble', 'v7_context', 'v9_catboost')
V12_SCOPED_RETURN_THRESHOLDS = (1.0, 1.1, 1.2)
V12_SCOPED_MIN_PROBABILITIES = (0.20, 0.30)
V12_SCOPED_TOP_N = (1,)
V12_SCOPED_VENUES = ('ST', 'HV')

# V13 acceleration and conservative policy coverage prior; tests remain exploratory.
# The exact same model profiles, seeds, thresholds and historical windows as V12.
# Set False for a strict baseline diagnostic if desired.
SHARE_HISTORY_FEATURE_PASSES = True
FAST_POLICY_SEARCH = True
MIN_PROJECTED_CONFIRMATION_BETS = 75  # buffer vs 50-bet confirmation gate
RUN_SHADOW_FOR_REJECTED = False  # skip unnecessary final-test model fits after pre-test rejection; True restores old shadow reports
CACHE_PARSED_RUNNERS = True
TIMING_REPORT = []

# Freeze this choice BEFORE running discovery. Do not switch based on test ROI.
# frozen: every test-period outcome is excluded from all test-period features.
# daily: only prior completed test dates update history; no model/policy refits.
HISTORY_MODE = 'daily'
# All derived running-position, times and margins are HISTORY-ONLY, never current predictors.
# Only verified BEFORE-bet snapshots may be used as current-run market features.
# The provided win_odds column may contain closing/post-race odds, so we deliberately
# use it only in *previous-run history*. The current-run win_odds is not a predictor.

BOOSTING_ITERATIONS = 200
RECENCY_HALF_LIFE_DAYS = 730
RETURN_THRESHOLDS = [1.00, 1.10, 1.20]
MIN_PROBABILITIES = [0.12, 0.20, 0.30]
TOP_N_OPTIONS = [1, 2]
MIN_FIT_DATES = 60
MIN_CALIBRATION_DATES = 20
MIN_VALIDATION_BETS = 150
MIN_BETS_PER_FOLD = 30
MIN_CONFIRMATION_BETS = 50
MIN_FINAL_TEST_BETS = 100
MIN_FOLD_ROI = 0.90
BOOTSTRAP_SAMPLES = 500
UNIT_STAKE = 500.0
SHOW_PLOTS = False  # faster Kaggle runtime; all CSV diagnostics are still saved
PLACE_POOL_TYPE = 'PLACE'
ALLOWED_LOCAL_RACECOURSES = {'ST', 'HV'}
ODDS_MERGE_KEYS = ['race_date', 'racecourse', 'race_no', 'pool_type', 'combination']
PLACE_ODDS_AUDIT_MODE = 'exclude'
def detailed_print(*args, **kwargs):
    pass



# V14: one predeclared return challenger; original probability models remain controls.
V14_VALUE_ENABLED = True
V14_VALUE_ITERATIONS = 150
V14_SCORE = 'value_logistic'
V14_POLICY_KEYS = ['model', 'score', 'min_return', 'min_probability', 'top_n', 'venue_filter']


def payout_weighted_rows(x, gross_returns, base_weights=None):
    """Loss -R log(q)-log(1-q) gives q/(1-q)=E[R|X], NOT PLACE hit probability."""
    r = np.asarray(gross_returns, dtype=float)
    if len(x) != len(r) or not np.isfinite(r).all() or (r < 0).any():
        raise ValueError('Require finite nonnegative audited gross-return labels.')
    w = np.ones(len(r)) if base_weights is None else np.asarray(base_weights, dtype=float)
    if len(w) != len(r) or not np.isfinite(w).all() or (w <= 0).any():
        raise ValueError('Require positive finite observation weights.')
    positive = r > 0
    return (np.concatenate([x, x[positive]], axis=0),
            np.concatenate([np.zeros(len(r)), np.ones(positive.sum())]),
            np.concatenate([w, w[positive] * r[positive]]))


def fit_value_logistic(xf, xc, xp, returns_fit, returns_cal, weights, category_mask):
    train_x, train_y, train_w = payout_weighted_rows(xf, returns_fit, weights)
    if np.unique(train_y).size < 2 or not np.any(np.asarray(returns_cal) > 0):
        return np.zeros(len(xp)), {'status': 'no_positive_historical_returns'}
    learner = HistGradientBoostingClassifier(
        max_iter=V14_VALUE_ITERATIONS, max_leaf_nodes=5, min_samples_leaf=120,
        learning_rate=0.035, l2_regularization=30., early_stopping=False,
        categorical_features=category_mask, random_state=42)
    learner.fit(train_x, train_y, sample_weight=train_w)
    raw_cal = logit(learner.predict_proba(xc)[:, 1])
    raw_future = logit(learner.predict_proba(xp)[:, 1])
    cx, cy, cw = payout_weighted_rows(raw_cal, returns_cal)
    calibrator = LogisticRegression(C=1., random_state=42).fit(cx, cy, sample_weight=cw)
    low, high = np.quantile(raw_cal[:, 0], [.01, .99])
    eta = calibrator.decision_function(np.clip(raw_future, low, high))
    prediction = np.exp(np.clip(eta, -20., 20.))
    return prediction, {
        'status': 'fitted', 'target': 'expected gross payout per unit stake; not hit probability',
        'fit_return_cap': None, 'calibration': 'historical payout-weighted logistic',
        'calibration_raw_logit_bounds': [float(low), float(high)],
        'calibration_slope': float(calibrator.coef_[0, 0]),
        'calibration_intercept': float(calibrator.intercept_[0]),
        'fit_rows_before_duplication': len(xf), 'fit_rows_after_duplication': len(train_x),
        'max_iter': V14_VALUE_ITERATIONS, 'max_leaf_nodes': 5,
        'min_samples_leaf': 120, 'l2_regularization': 30.}


def paired_weekly_roi_difference(challenger, baseline, prediction_dates):
    """Paired calendar-week resampling, development only; descriptive CI."""
    weeks = pd.Index(pd.to_datetime(prediction_dates).dt.to_period('W').astype(str).unique())
    if len(weeks) < 8:
        return {'lower90_difference': np.nan, 'median_difference': np.nan, 'weeks': len(weeks)}
    def totals(detail):
        frame = detail.loc[detail['buy_flag']].copy()
        frame['week'] = pd.to_datetime(frame['race_date']).dt.to_period('W').astype(str)
        return frame.groupby('week')[['stake', 'payout']].sum().reindex(weeks, fill_value=0).to_numpy()
    c, b = totals(challenger), totals(baseline)
    ix = np.random.RandomState(140).randint(0, len(weeks), size=(BOOTSTRAP_SAMPLES, len(weeks)))
    cs, bs = c[ix].sum(axis=1), b[ix].sum(axis=1)
    valid = (cs[:, 0] > 0) & (bs[:, 0] > 0)
    if valid.sum() < max(10, int(.9 * BOOTSTRAP_SAMPLES)):
        return {'lower90_difference': np.nan, 'median_difference': np.nan, 'weeks': len(weeks)}
    delta = cs[valid, 1]/cs[valid, 0] - bs[valid, 1]/bs[valid, 0]
    return {'lower90_difference': float(np.quantile(delta, .10)),
            'median_difference': float(np.median(delta)), 'weeks': len(weeks)}


def validation_search(oof, outcomes):
    _, _, table = _v13_validation_search(oof, outcomes)
    baseline = table.loc[table['score'].ne(V14_SCORE)].iloc[0]
    challengers = table.loc[table['score'].eq(V14_SCORE) & table['eligible']]
    selected, reason = baseline, 'retain_v13_control'
    def policy(row):
        p = {key: row[key] for key in V14_POLICY_KEYS}
        p['top_n'] = int(p['top_n'])
        return p
    comparison = {'baseline_policy': policy(baseline),
                  'baseline_discovery_roi': float(baseline['roi']),
                  'baseline_eligible': bool(baseline['eligible']),
                  'challenger_eligible_count': len(challengers)}
    if len(challengers):
        challenger = challengers.iloc[0]
        comparison['challenger_policy'] = policy(challenger)
        comparison['challenger_discovery_roi'] = float(challenger['roi'])
        if not bool(baseline['eligible']):
            selected, reason = challenger, 'challenger_passed_original_gates_control_failed'
        else:
            c = settle_plan(make_bet_plan(oof, policy(challenger)), outcomes)
            b = settle_plan(make_bet_plan(oof, policy(baseline)), outcomes)
            difference = paired_weekly_roi_difference(c, b, oof['race_date'])
            comparison.update(difference)
            if difference['lower90_difference'] > 0:
                selected, reason = challenger, 'challenger_passed_gates_and_paired_comparison'
            else:
                reason = 'retain_v13_control_no_clear_development_improvement'
    comparison['decision'] = reason
    comparison['selected_policy'] = policy(selected)
    comparison['interval_caveat'] = 'descriptive after selection, not multiplicity-adjusted; confirmation still required'
    table['selected_for_confirmation'] = table.index == selected.name
    table['v14_selection_reason'] = reason
    table.attrs['v14_comparison'] = comparison
    print('V14 development comparison:', comparison, flush=True)
    return policy(selected), bool(selected['eligible']), table


def save_return_decomposition(details, directory, stage):
    """Fixed-unit forecasts: E[R]=P(hit)*E[paid multiple|hit]. Diagnostics only."""
    b = details.loc[details['buy_flag']].copy()
    if b.empty:
        report = {'bought_tickets': 0, 'note': 'no purchases'}
    else:
        ph = float(b['predicted_place_probability'].mean())
        rh = float(b['hit_bought'].mean())
        pr = float(b['predicted_return'].mean())
        rr = float(b['payout'].sum()/b['stake'].sum())
        report = {'bought_tickets': len(b), 'hit_tickets': int(b['hit_bought'].sum()),
                  'mean_predicted_hit_probability': ph, 'actual_hit_rate': rh,
                  'mean_predicted_gross_return': pr, 'actual_gross_roi': rr,
                  'implied_probability_weighted_paid_multiple': pr/ph if ph > 0 else np.nan,
                  'actual_average_paid_multiple_on_hits': rr/rh if rh > 0 else np.nan,
                  'total_stake': float(b['stake'].sum()), 'total_payout': float(b['payout'].sum()),
                  'total_profit': float(b['profit'].sum()),
                  'unresolved_bought_tickets': int((~b['audit_ok']).sum())}
    write_json(Path(directory)/f'{stage}_return_decomposition.json', report)
    print(stage, 'hit/payout decomposition:', report, flush=True)



def parse_finishing_position(value):
    """
    着順の先頭にある数値を取得する。

    例: 2, "2", "2 DH", "2DH", "2 (DH)" -> 2
    WV / PU / DNFなど、数値着順を持たない値はNaNのままにする。
    """
    if pd.isna(value):
        return np.nan
    if isinstance(value, (int, float, np.integer, np.floating)):
        return float(value) if np.isfinite(value) else np.nan

    match = re.match(r"^\s*(\d+(?:\.\d+)?)", str(value))
    if match is None:
        return np.nan
    return float(match.group(1))

def normalize_racecourse_for_odds(x):
    s = str(x).strip().upper()

    if s in ["ST", "SHA TIN", "SHATIN", "SHA_TIN"]:
        return "ST"

    if s in ["HV", "HAPPY VALLEY", "HAPPYVALLEY", "HAPPY_VALLEY"]:
        return "HV"

    return s

def normalize_boolean_flag(value, column_name):
    normalized = str(value).strip().upper()
    true_values = {"TRUE", "T", "1", "YES", "Y"}
    false_values = {"FALSE", "F", "0", "NO", "N"}
    if normalized in true_values:
        return True
    if normalized in false_values:
        return False
    raise ValueError(
        f"{column_name}に解釈できない値があります: {value!r}"
    )

def normalize_place_combination_for_eval(value):
    if pd.isna(value):
        return ""
    text_value = str(value).strip()
    if not text_value or any(separator in text_value for separator in [",", "/", "-"]):
        return ""
    try:
        numeric_value = float(text_value)
    except (TypeError, ValueError):
        return ""
    if not numeric_value.is_integer() or numeric_value <= 0:
        return ""
    return str(int(numeric_value))

def normalize_odds_for_eval(odds_df):
    """
    parser v2の明示メタデータを使い、LOCALのPLACE配当だけを安全に残す。

    race_noの並びから海外開催を推定しない。section_codeとis_overseasが
    矛盾する場合、LOCALなのにST/HV以外の場合、非numeric配当、単位不明、
    結合キー重複がある場合は、誤って損失計上せず処理を停止する。
    """
    odds_eval = odds_df.copy().reset_index(drop=True)
    required_cols = {
        "race_date",
        "section_code",
        "is_overseas",
        "racecourse",
        "race_no",
        "pool_type",
        "combination",
        "dividend_hkd",
        "dividend_unit_hkd",
        "row_status",
    }
    missing_cols = required_cols - set(odds_eval.columns)
    if missing_cols:
        raise ValueError(
            "安全なodds判定に必要なparser v2列がありません: "
            + str(sorted(missing_cols))
        )

    input_rows = len(odds_eval)
    odds_eval["pool_type"] = (
        odds_eval["pool_type"]
        .astype(str)
        .str.strip()
        .str.upper()
    )
    odds_eval = odds_eval.loc[
        odds_eval["pool_type"].eq(PLACE_POOL_TYPE)
    ].copy()
    if odds_eval.empty:
        raise ValueError("oddsファイルにpool_type='PLACE'の行がありません")

    odds_eval["section_code"] = (
        odds_eval["section_code"]
        .astype(str)
        .str.strip()
        .str.upper()
    )
    odds_eval["is_overseas_bool"] = odds_eval["is_overseas"].apply(
        lambda value: normalize_boolean_flag(value, "is_overseas")
    )
    section_is_local = odds_eval["section_code"].eq("LOCAL")
    metadata_mismatch = section_is_local.ne(~odds_eval["is_overseas_bool"])
    if metadata_mismatch.any():
        mismatch_cols = [
            col for col in [
                "race_date", "section_code", "section_label",
                "section_country", "is_overseas", "racecourse",
                "race_no", "pool_type", "source_url",
            ]
            if col in odds_eval.columns
        ]
        print("\nsection_codeとis_overseasが矛盾するPLACE行:")
        print(
            odds_eval.loc[metadata_mismatch, mismatch_cols]
            .head(30)
            .to_string(index=False)
        )
        raise ValueError(
            "section_codeとis_overseasが矛盾しています。"
            "開催区分を推測せず処理を停止します。"
        )

    removed_overseas_rows = int((~section_is_local).sum())
    place_rows_before_local_filter = len(odds_eval)
    odds_eval = odds_eval.loc[
        section_is_local & ~odds_eval["is_overseas_bool"]
    ].copy()
    if odds_eval.empty:
        raise ValueError("LOCALかつis_overseas=FALSEのPLACE行がありません")

    odds_eval["race_date"] = pd.to_datetime(
        odds_eval["race_date"], errors="coerce"
    ).dt.strftime("%Y-%m-%d")
    odds_eval["racecourse"] = odds_eval["racecourse"].apply(
        normalize_racecourse_for_odds
    )
    unknown_local_courses = sorted(
        set(odds_eval["racecourse"].dropna().unique())
        - set(ALLOWED_LOCAL_RACECOURSES)
    )
    if unknown_local_courses:
        display_cols = [
            col for col in [
                "race_date", "section_code", "is_overseas",
                "racecourse", "race_no", "source_url",
            ]
            if col in odds_eval.columns
        ]
        print("\nLOCAL扱いなのに未知の競馬場があるPLACE行:")
        print(
            odds_eval.loc[
                odds_eval["racecourse"].isin(unknown_local_courses),
                display_cols,
            ]
            .drop_duplicates()
            .head(30)
            .to_string(index=False)
        )
        raise ValueError(
            "LOCALのPLACE行にST/HV以外の競馬場があります: "
            + str(unknown_local_courses)
        )

    odds_eval["race_no"] = pd.to_numeric(
        odds_eval["race_no"], errors="coerce"
    ).astype("Int64")
    odds_eval["combination"] = odds_eval["combination"].apply(
        normalize_place_combination_for_eval
    )
    odds_eval["dividend_hkd"] = pd.to_numeric(
        odds_eval["dividend_hkd"], errors="coerce"
    )
    odds_eval["dividend_unit_hkd"] = pd.to_numeric(
        odds_eval["dividend_unit_hkd"], errors="coerce"
    )
    odds_eval["row_status"] = (
        odds_eval["row_status"]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    invalid_mask = (
        odds_eval["race_date"].isna()
        | odds_eval["racecourse"].eq("")
        | odds_eval["race_no"].isna()
        | odds_eval["race_no"].le(0)
        | odds_eval["combination"].eq("")
        | odds_eval["dividend_hkd"].isna()
        | odds_eval["dividend_hkd"].le(0)
        | odds_eval["dividend_unit_hkd"].isna()
        | odds_eval["dividend_unit_hkd"].le(0)
        | ~odds_eval["row_status"].eq("numeric")
    )
    if invalid_mask.any():
        invalid_cols = [
            col for col in [
                "race_date", "racecourse", "race_no", "pool_type",
                "combination", "dividend_hkd", "dividend_unit_hkd",
                "row_status", "source_url",
            ]
            if col in odds_eval.columns
        ]
        print("\n安全に評価できないLOCAL PLACE行:")
        print(
            odds_eval.loc[invalid_mask, invalid_cols]
            .head(30)
            .to_string(index=False)
        )
        raise ValueError(
            f"安全に評価できないLOCAL PLACE行が"
            f"{int(invalid_mask.sum())}件あります"
        )

    exact_duplicate_cols = (
        ODDS_MERGE_KEYS + ["dividend_hkd", "dividend_unit_hkd"]
    )
    exact_duplicate_mask = odds_eval.duplicated(
        subset=exact_duplicate_cols,
        keep="first",
    )
    num_exact_duplicates = int(exact_duplicate_mask.sum())
    odds_eval = odds_eval.loc[~exact_duplicate_mask].copy()

    conflicting_duplicate_mask = odds_eval.duplicated(
        subset=ODDS_MERGE_KEYS,
        keep=False,
    )
    if conflicting_duplicate_mask.any():
        display_cols = ODDS_MERGE_KEYS + [
            "dividend_hkd", "dividend_unit_hkd"
        ]
        print("\nLOCAL PLACE oddsに競合する重複があります:")
        print(
            odds_eval.loc[conflicting_duplicate_mask, display_cols]
            .sort_values(ODDS_MERGE_KEYS)
            .to_string(index=False)
        )
        raise ValueError(
            "LOCAL PLACE oddsの結合キーが一意ではありません"
        )

    odds_eval = odds_eval.reset_index(drop=True)
    detailed_print("\n" + "=" * 70)
    detailed_print("Safe PLACE odds cleaning summary")
    detailed_print("input rows:", input_rows)
    detailed_print(
        "PLACE rows before LOCAL filter:",
        place_rows_before_local_filter,
    )
    detailed_print("removed overseas PLACE rows:", removed_overseas_rows)
    detailed_print("removed exact duplicate rows:", num_exact_duplicates)
    detailed_print("safe LOCAL PLACE rows:", len(odds_eval))
    detailed_print(
        "local racecourses:",
        sorted(odds_eval["racecourse"].unique()),
    )
    detailed_print(
        "dividend units:",
        sorted(odds_eval["dividend_unit_hkd"].unique()),
    )
    return odds_eval

def validate_place_odds_against_results(prediction_result, odds_place, label):
    """
    各レースの実着順から得られる複勝対象馬番と、CSVのPLACE配当馬番を照合する。

    PLACE配当行が2件なら実着順2位以内、3件以上なら3位以内を対象とする。
    同着で配当馬が4頭以上でも順位上限は3のまま照合する。
    """
    if PLACE_ODDS_AUDIT_MODE not in {"exclude", "raise"}:
        raise ValueError(
            "PLACE_ODDS_AUDIT_MODEは'exclude'または'raise'にしてください"
        )

    result = prediction_result.copy().reset_index(drop=True)
    result["race_date"] = pd.to_datetime(
        result["race_date"], errors="coerce"
    ).dt.strftime("%Y-%m-%d")
    result["racecourse"] = result["racecourse"].apply(
        normalize_racecourse_for_odds
    )
    result["race_no"] = pd.to_numeric(
        result["race_no"], errors="coerce"
    ).astype("Int64")
    result["pool_type"] = PLACE_POOL_TYPE
    result["_horse_number"] = pd.to_numeric(
        result["horse_number"], errors="coerce"
    )
    result["_actual_finish"] = pd.to_numeric(
        result["finishing_position_num"], errors="coerce"
    )

    result_invalid = (
        result["race_date"].isna()
        | ~result["racecourse"].isin(ALLOWED_LOCAL_RACECOURSES)
        | result["race_no"].isna()
        | result["_horse_number"].isna()
        | result["_actual_finish"].isna()
    )
    if result_invalid.any():
        raise ValueError(
            f"{label}: 実結果側のPLACE照合キーまたは着順に欠損があります"
        )

    race_keys = ["race_date", "racecourse", "race_no", "pool_type"]
    odds_groups = {
        key: group.copy()
        for key, group in odds_place.groupby(race_keys, sort=False)
    }
    audit_rows = []

    for key, race_group in result.groupby(race_keys, sort=False):
        odds_group = odds_groups.get(key)
        if odds_group is None or odds_group.empty:
            # 配当行がない場合は複勝対象頭数を確定できない。診断用に
            # 実着順3位以内を記録するが、このレースは評価には使わない。
            actual_top3_horses = {
                int(horse_number)
                for horse_number, finish in zip(
                    race_group["_horse_number"],
                    race_group["_actual_finish"],
                )
                if float(finish) <= 3
            }
            audit_rows.append({
                "race_date": key[0],
                "racecourse": key[1],
                "race_no": int(key[2]),
                "pool_type": PLACE_POOL_TYPE,
                "place_paid_horse_count": 0,
                "place_paid_rank_limit": np.nan,
                "actual_paid_horses": ",".join(
                    map(str, sorted(actual_top3_horses))
                ),
                "odds_paid_horses": "",
                "actual_only_horses": ",".join(
                    map(str, sorted(actual_top3_horses))
                ),
                "odds_only_horses": "",
                "place_odds_audit_ok": False,
                "place_odds_audit_reason": "PLACE odds race missing",
            })
            continue

        odds_paid_horses = {
            int(value)
            for value in odds_group["combination"].astype(str)
        }
        paid_rank_limit = min(3, len(odds_paid_horses))
        actual_paid_horses = {
            int(horse_number)
            for horse_number, finish in zip(
                race_group["_horse_number"],
                race_group["_actual_finish"],
            )
            if float(finish) <= paid_rank_limit
        }
        actual_only_horses = actual_paid_horses - odds_paid_horses
        odds_only_horses = odds_paid_horses - actual_paid_horses
        audit_ok = not actual_only_horses and not odds_only_horses
        audit_rows.append({
            "race_date": key[0],
            "racecourse": key[1],
            "race_no": int(key[2]),
            "pool_type": PLACE_POOL_TYPE,
            "place_paid_horse_count": len(odds_paid_horses),
            "place_paid_rank_limit": paid_rank_limit,
            "actual_paid_horses": ",".join(
                map(str, sorted(actual_paid_horses))
            ),
            "odds_paid_horses": ",".join(
                map(str, sorted(odds_paid_horses))
            ),
            "actual_only_horses": ",".join(
                map(str, sorted(actual_only_horses))
            ),
            "odds_only_horses": ",".join(
                map(str, sorted(odds_only_horses))
            ),
            "place_odds_audit_ok": bool(audit_ok),
            "place_odds_audit_reason": (
                "" if audit_ok else "paid horse set mismatch"
            ),
        })

    audit_df = pd.DataFrame(audit_rows)
    if audit_df.empty:
        raise ValueError(f"{label}: PLACE照合対象レースがありません")

    failed = audit_df.loc[~audit_df["place_odds_audit_ok"]].copy()
    if not failed.empty:
        display_cols = [
            "race_date", "racecourse", "race_no",
            "place_odds_audit_reason",
            "actual_paid_horses", "odds_paid_horses",
            "actual_only_horses", "odds_only_horses",
        ]
        print("\n実着順とPLACE配当馬番が一致しないレース:")
        print(failed[display_cols].head(30).to_string(index=False))

        if PLACE_ODDS_AUDIT_MODE == "raise":
            raise ValueError(
                f"{label}: 実着順とPLACE oddsが一致しないレースが"
                f"{len(failed)}件あります。"
                "競馬場・race_no・開催区分の対応を確認してください。"
            )

        print(
            f"{label}: 監査NGの{len(failed)}レースは、"
            "損失扱いせず全thresholdの評価対象から除外します。"
        )

    return audit_df

# All features below are explicitly generated from pre-race fields or older outcomes.
PRE_RACE_NUMERIC = ['actual_weight_num', 'declared_horse_weight_num', 'draw_num', 'race_distance', 'horse_rating_num', 'horse_age_num']
PRE_RACE_CATEGORICAL = ['race_course', 'race_class', 'track_condition', 'track', 'jockey', 'trainer', 'horse_gear', 'horse_type']
IDENTIFIERS = ['row_id', 'race_id', 'race_date', 'racecourse', 'race_no', 'horse_number']


def prepare_runners(raw):
    data = raw.copy()
    required = {'horse_id', 'horse_number', 'race_id', 'race_date_iso', 'race_course', 'finishing_position'}
    if not required.issubset(data.columns):
        raise ValueError('Missing runner columns: ' + str(sorted(required - set(data.columns))))
    data['race_date'] = pd.to_datetime(data['race_date_iso'], errors='raise').dt.normalize()
    data['racecourse'] = data['race_course'].map(normalize_racecourse_for_odds)
    data = data.loc[data['racecourse'].isin(ALLOWED_LOCAL_RACECOURSES)].copy()
    # Unnumbered reserve records cannot be selected as betting combinations.
    data = data.loc[pd.to_numeric(data['horse_number'], errors='coerce').notna()].copy()
    race_number_col = 'race_number' if 'race_number' in data else 'race_no'
    if race_number_col not in data:
        raise ValueError('Explicit race_number/race_no is required; race order is not inferred.')
    data['race_no'] = pd.to_numeric(data[race_number_col], errors='raise').astype(int)
    data['horse_number'] = pd.to_numeric(data['horse_number'], errors='raise').astype(int)
    if data[['horse_id', 'race_id', 'race_date']].isna().any().any():
        raise ValueError('Missing horse/race/date identifiers.')
    status = data['finishing_position'].astype(str).str.strip().str.upper()
    # Withdrawn runners are excluded. Nonfinishers remain eligible runners/losses.
    withdrawn = status.isin({'WV', 'WV-A', 'WV-T', 'WX', 'WX-A', 'WXNR', 'WD', 'SCR', 'WDR', 'WITHDRAWN'})
    data = data.loc[~withdrawn].copy()
    status = status.loc[data.index]
    finish = data['finishing_position'].map(parse_finishing_position)
    nonfinish = status.isin({'DNF', 'PU', 'UR', 'F', 'FE', 'DQ', 'DISQ', 'TNP'})
    data['uncertain_settlement'] = status.eq('TNP')
    unknown = finish.isna() & ~nonfinish
    if unknown.any():
        raise ValueError('Unrecognized finishing status; review before running: ' + str(status[unknown].unique()))
    data['finishing_position_num'] = finish.fillna(999).astype(float)
    if data['finishing_position_num'].le(0).any():
        raise ValueError('Finishing positions must be positive.')
    data['horse_id'] = data['horse_id'].astype(str)
    data['race_id'] = data['race_id'].astype(str)
    keys = ['race_date', 'racecourse', 'race_no']
    if data.duplicated(keys + ['horse_number']).any():
        raise ValueError('Duplicate runners; resolve duplicates before evaluation.')
    if data.groupby('race_id')[keys].nunique().gt(1).any().any():
        raise ValueError('race_id must identify exactly one race.')
    data['field_size'] = data.groupby('race_id')['horse_number'].transform('size')
    # Explicit pre-race whitelist: ratings, age, weights and draw are known before
    # the race. Current-race odds (which may be captured at/after the close), times,
    # running positions and finishing margins are NOT direct features.
    if 'horse_rating' in data:
        data['horse_rating_num'] = pd.to_numeric(data['horse_rating'], errors='coerce')
    if 'horse_age' in data:
        data['horse_age_num'] = pd.to_numeric(data['horse_age'], errors='coerce')
    data['_lagged_win_odds_source'] = (pd.to_numeric(data['win_odds'], errors='coerce')
                                      if 'win_odds' in data else np.nan)
    # These outcome-derived inputs are a quarantined HISTORY SOURCE: NEVER read
    # by transform(), training matrix creation, or prediction for the focal race.
    # They only enter PastOnlyFeatures.update() after a race has finished.
    history_only_cols = {
        '_past_first_call_source': 'first_call_position',
        '_past_last_call_source': 'last_call_position',
        '_past_time_source': 'finish_time_seconds',
        '_past_margin_source': 'length_behind_winner_num',
        '_past_avg_running_source': 'avg_running_position',
    }
    for dest, source in history_only_cols.items():
        data[dest] = (pd.to_numeric(data[source], errors='coerce')
                      if source in data else np.nan)
    # Numerous malformed raw margin strings were converted to large Excel serial
    # numbers (e.g. 37990). Those are NOT lengths; discard rather than clipping.
    data['_past_margin_source'] = data['_past_margin_source'].where(
        data['_past_margin_source'].between(0, 40))
    data['_past_time_source'] = data['_past_time_source'].where(
        data['_past_time_source'].between(30, 260))
    for col in ['_past_first_call_source', '_past_last_call_source', '_past_avg_running_source']:
        data[col] = data[col].where(data[col].between(1, 24))
    # The class is listed pre-race, unlike the result/tempo columns above.
    data['race_class_num'] = pd.to_numeric(
        data['race_class'].astype(str).str.extract(r'(?i)class\s*(\d+)', expand=False),
        errors='coerce') if 'race_class' in data else np.nan
    for col in PRE_RACE_NUMERIC:
        data[col] = pd.to_numeric(data[col], errors='coerce') if col in data else np.nan
    for col in PRE_RACE_CATEGORICAL:
        data[col] = data[col].fillna('missing').astype(str) if col in data else 'missing'
    data = data.sort_values(keys + ['horse_number'], kind='mergesort').reset_index(drop=True)
    data['row_id'] = np.arange(len(data))
    return data


def attach_outcomes(runners, odds, label):
    """Called on development outcomes, and on final test ONLY after locking bets."""
    data = runners.copy()
    audit = validate_place_odds_against_results(data, odds, label)
    uncertain_races = set(data.loc[data['uncertain_settlement'], 'race_id'])
    uncertain_keys = set(map(tuple, data.loc[data['race_id'].isin(uncertain_races),
                         ['race_date', 'racecourse', 'race_no']].assign(
                             race_date=lambda x: x['race_date'].dt.strftime('%Y-%m-%d')).to_numpy()))
    uncertain_mask = audit[['race_date', 'racecourse', 'race_no']].apply(tuple, axis=1).isin(uncertain_keys)
    audit.loc[uncertain_mask, 'place_odds_audit_ok'] = False
    audit.loc[uncertain_mask, 'place_odds_audit_reason'] = 'TNP settlement/refund requires explicit evidence'
    data['race_date'] = data['race_date'].dt.strftime('%Y-%m-%d')
    data = data.merge(audit[['race_date', 'racecourse', 'race_no', 'place_odds_audit_ok']],
                      on=['race_date', 'racecourse', 'race_no'], how='left', validate='many_to_one')
    data['combination'] = data['horse_number'].astype(str)
    data['pool_type'] = PLACE_POOL_TYPE
    data = data.merge(odds[ODDS_MERGE_KEYS + ['dividend_hkd', 'dividend_unit_hkd']],
                      on=ODDS_MERGE_KEYS, how='left', validate='many_to_one')
    if data['place_odds_audit_ok'].isna().any():
        raise ValueError('Missing audit result.')
    data['audit_ok'] = data['place_odds_audit_ok'].astype(bool)
    hit = data['dividend_hkd'].notna()
    data['place_hit'] = hit.astype(float).where(data['audit_ok'])
    data['return_target'] = (data['dividend_hkd'] / data['dividend_unit_hkd']).fillna(0).where(data['audit_ok'])
    known_returns = data.loc[data['audit_ok'], 'return_target'].to_numpy()
    if not np.isfinite(known_returns).all() or (known_returns < 0).any():
        raise ValueError('Audited payout ratios must be finite and nonnegative.')
    data['race_date'] = pd.to_datetime(data['race_date'])
    return data.sort_values('row_id').reset_index(drop=True), audit


class PastOnlyFeatures:
    """Race-date snapshots: same-day outcomes never inform same-day predictions.

    Training traverses each date in order. The block builder either freezes
    evaluation histories or advances them after each completed date. Current-race result, dividends and odds
    are never read by ``transform``. Historical market odds are read only when
    ``update`` is called for races that have already occurred.
    """
    def __init__(self):
        self.horses = {}
        self.groups = {}
        self.value_groups = {}  # audited returns from races strictly before the prediction date
        self.last_date = None

    @staticmethod
    def group_keys(row):
        distance = round(float(row['race_distance']) / 200) if np.isfinite(row['race_distance']) else -1
        return [
            ('jockey', row['jockey']), ('trainer', row['trainer']),
            ('horse_course', (row['horse_id'], row['racecourse'])),
            ('horse_distance', (row['horse_id'], distance)),
            ('jockey_trainer', (row['jockey'], row['trainer'])),
            ('trainer_course', (row['trainer'], row['racecourse'])),
            ('jockey_course', (row['jockey'], row['racecourse'])),
        ]

    @staticmethod
    def avg(history, key):
        values = [r[key] for r in history if pd.notna(r.get(key))]
        return float(np.mean(values)) if values else np.nan

    def transform(self, runners):
        if self.last_date is not None and runners['race_date'].min() <= self.last_date:
            raise ValueError('History must end strictly before every prediction date.')
        records = []
        for row in runners.to_dict('records'):
            horse = self.horses.get(row['horse_id'], [])
            f = {'row_id': row['row_id']}
            for col in PRE_RACE_NUMERIC + PRE_RACE_CATEGORICAL:
                f['f_' + col] = row[col]
            f['f_field_size'] = row['field_size']
            f['f_draw_fraction'] = ((row['draw_num'] - 1) / max(1, row['field_size'] - 1)
                                    if np.isfinite(row['draw_num']) else np.nan)
            f['f_weight_ratio'] = (row['actual_weight_num'] / row['declared_horse_weight_num']
                                   if row['declared_horse_weight_num'] > 0 else np.nan)
            f['f_month_sin'] = np.sin(2 * np.pi * row['race_date'].month / 12)
            f['f_month_cos'] = np.cos(2 * np.pi * row['race_date'].month / 12)
            # Current listed class/track are pre-race facts. No current result data.
            f['f_class_level'] = row.get('race_class_num', np.nan)
            f['f_current_rail_track'] = row.get('track', 'missing')
            f['f_horse_runs'] = len(horse)
            f['f_days_since_run'] = (row['race_date'] - horse[-1]['date']).days if horse else np.nan
            f['f_horse_top3_bayes'] = (sum(r['top3'] for r in horse) + 2.5) / (len(horse) + 10)
            f['f_horse_wins_bayes'] = (sum(r['won'] for r in horse) + 0.8) / (len(horse) + 10)
            # Historical *settled* PLACE gross returns: never current-race dividends.
            # Smooth heavily because payouts have high variance and change with market prices.
            horse_returns = [r['gross_return'] for r in horse if np.isfinite(r['gross_return'])]
            f['f_horse_prior_place_roi'] = (sum(horse_returns) + 0.85 * 15) / (len(horse_returns) + 15)
            f['f_horse_audited_return_runs'] = len(horse_returns)
            last_returns = horse_returns[-10:]
            f['f_horse_recent_place_roi'] = (sum(last_returns) + 0.85 * 8) / (len(last_returns) + 8)
            for window in [3, 5, 10]:
                recent = horse[-window:]
                f[f'f_top3_last{window}'] = self.avg(recent, 'top3')
                f[f'f_finish_score_last{window}'] = self.avg(recent, 'score')
                f[f'f_win_rate_last{window}'] = self.avg(recent, 'won')
                f[f'f_winning_odds_last{window}'] = self.avg(recent, 'win_odds')
            # Rolling form and race-condition strength, constructed strictly from
            # completed races. The focal race is never in `horse` at this point.
            for window in [3, 5, 10]:
                h = horse[-window:]
                if h:
                    recency_weights = np.arange(1, len(h) + 1, dtype=float)
                    f[f'f_score_recency{window}'] = float(np.average([r['score'] for r in h], weights=recency_weights))
                    f[f'f_top3_recency{window}'] = float(np.average([r['top3'] for r in h], weights=recency_weights))
                    f[f'f_score_volatility{window}'] = float(np.std([r['score'] for r in h]))
                else:
                    f[f'f_score_recency{window}'] = np.nan
                    f[f'f_top3_recency{window}'] = np.nan
                    f[f'f_score_volatility{window}'] = np.nan
            f['f_prev_finish_score'] = horse[-1]['score'] if horse else np.nan
            f['f_prev_winning_odds'] = horse[-1]['win_odds'] if horse else np.nan
            known_odds = [r['win_odds'] for r in horse[-5:] if np.isfinite(r['win_odds']) and r['win_odds'] > 0]
            f['f_lagged_win_market_strength'] = float(np.mean([1.0 / x for x in known_odds])) if known_odds else np.nan
            f['f_prev_weight'] = horse[-1]['weight'] if horse else np.nan
            f['f_weight_change'] = (row['actual_weight_num'] - horse[-1]['weight']
                                    if horse else np.nan)
            f['f_distance_change'] = (row['race_distance'] - horse[-1]['distance']
                                      if horse else np.nan)
            f['f_prev_distance'] = horse[-1]['distance'] if horse else np.nan
            # Strength against similar past conditions (including course, track,
            # distance and class), never looking at this race's result.
            simdist = [r for r in horse if abs(r['distance'] - row['race_distance']) <= 200]
            samecourse = [r for r in horse if r['racecourse'] == row['racecourse']]
            f['f_similar_distance_runs'] = len(simdist)
            f['f_similar_distance_form'] = self.avg(simdist[-6:], 'score')
            f['f_same_course_form'] = self.avg(samecourse[-6:], 'score')
            f['f_same_course_runs'] = len(samecourse)
            f['f_course_distance_form'] = self.avg(
                [r for r in simdist if r['racecourse'] == row['racecourse']][-6:], 'score')
            f['f_similar_distance_place_rate'] = self.avg(simdist[-8:], 'top3')
            f['f_same_course_place_rate'] = self.avg(samecourse[-8:], 'top3')
            f['f_distance_place_bayes'] = (sum(r['top3'] for r in simdist) + 2.5) / (len(simdist) + 10)
            # NEW V6: running style, speed and finishing performance from prior
            # completed races, expressed relative to the old field size.
            recent3 = horse[-3:]
            recent5 = horse[-5:]
            recent10 = horse[-10:]
            f['f_prior_early_position'] = self.avg(recent5, 'early_fraction')
            f['f_prior_late_position'] = self.avg(recent5, 'late_fraction')
            f['f_prior_avg_running_position'] = self.avg(recent5, 'avg_running_fraction')
            f['f_prior_late_gain'] = self.avg(recent5, 'late_gain_fraction')
            f['f_prior_late_gain_last3'] = self.avg(recent3, 'late_gain_fraction')
            f['f_prior_time_margin_sec_per_km'] = self.avg(recent5, 'time_margin_per_km')
            f['f_prior_time_margin_last3'] = self.avg(recent3, 'time_margin_per_km')
            f['f_prior_length_margin'] = self.avg(recent5, 'margin_lengths')
            f['f_prior_length_margin_last3'] = self.avg(recent3, 'margin_lengths')
            f['f_prior_fast_finishing_rate'] = self.avg(recent10, 'fast_finish')
            f['f_prior_front_running_rate'] = self.avg(recent10, 'front_runner')
            f['f_recent_speed_improvement'] = (self.avg(recent3, 'time_margin_per_km') -
                                               self.avg(horse[-8:-3], 'time_margin_per_km')
                                               if len(horse) >= 6 else np.nan)
            last_weight = [r['bodyweight'] for r in recent5 if np.isfinite(r['bodyweight'])]
            f['f_bodyweight_prior_mean5'] = float(np.mean(last_weight)) if last_weight else np.nan
            f['f_bodyweight_change'] = (row['declared_horse_weight_num'] - last_weight[-1]
                                           if last_weight and np.isfinite(row['declared_horse_weight_num'])
                                           else np.nan)
            f['f_bodyweight_change_relative'] = (f['f_bodyweight_change'] / last_weight[-1]
                                                    if last_weight and last_weight[-1] > 0 else np.nan)
            last_carried = [r['carried_weight'] for r in recent3 if np.isfinite(r['carried_weight'])]
            f['f_carried_weight_change'] = (row['actual_weight_num'] - last_carried[-1]
                                              if last_carried and np.isfinite(row['actual_weight_num']) else np.nan)
            last_class = [r['class_level'] for r in recent5 if np.isfinite(r['class_level'])]
            f['f_previous_class_level'] = last_class[-1] if last_class else np.nan
            f['f_class_move'] = (row['race_class_num'] - last_class[-1]
                                  if last_class and np.isfinite(row['race_class_num']) else np.nan)
            same_class = [r for r in horse if r['class_level'] == row['race_class_num']]
            f['f_same_class_runs'] = len(same_class)
            f['f_same_class_form'] = self.avg(same_class[-8:], 'score')
            surface = [r for r in horse if r['track'] == row['track']]
            f['f_same_track_runs'] = len(surface)
            f['f_same_track_form'] = self.avg(surface[-8:], 'score')
            f['f_same_track_place_rate'] = self.avg(surface[-8:], 'top3')
            f['f_early_draw_interaction'] = (f['f_prior_early_position'] * f['f_draw_fraction']
                                                if np.isfinite(f['f_prior_early_position'])
                                                and np.isfinite(f['f_draw_fraction']) else np.nan)
            f['f_recent_finish_vs_early'] = (f['f_prior_early_position'] -
                                               self.avg(recent5, 'finish_fraction')
                                               if np.isfinite(f['f_prior_early_position']) else np.nan)
            f['f_course_place_bayes'] = (sum(r['top3'] for r in samecourse) + 2.5) / (len(samecourse) + 10)
            f['f_improving_form'] = (self.avg(horse[-3:], 'score') - self.avg(horse[-10:-3], 'score')
                                     if len(horse) >= 5 else np.nan)
            for role, key in self.group_keys(row):
                successes, runs = self.groups.get((role, key), (0, 0))
                f['f_' + role + '_runs'] = runs
                f['f_' + role + '_top3_rate'] = (successes + 2.5) / (runs + 10)
                return_sum, audited_runs = self.value_groups.get((role, key), (0.0, 0))
                f['f_' + role + '_roi_shrunk'] = (return_sum + 0.85 * 35) / (audited_runs + 35)
            records.append(f)
        features = pd.DataFrame(records).set_index('row_id')
        race_ids = runners.set_index('row_id')['race_id'].reindex(features.index)
        for col in ['f_top3_last5', 'f_finish_score_last5', 'f_actual_weight_num',
                    'f_draw_fraction', 'f_horse_rating_num', 'f_horse_top3_bayes',
                    'f_similar_distance_form', 'f_jockey_top3_rate', 'f_trainer_top3_rate',
                    'f_score_recency5', 'f_top3_recency5',
                    'f_course_place_bayes', 'f_distance_place_bayes',
                    'f_lagged_win_market_strength', 'f_horse_prior_place_roi',
                    'f_jockey_roi_shrunk', 'f_trainer_roi_shrunk',
                    'f_prior_early_position', 'f_prior_late_gain',
                    'f_prior_time_margin_sec_per_km', 'f_prior_length_margin',
                    'f_bodyweight_change', 'f_class_move',
                    'f_same_class_form', 'f_same_track_form'] :

            if col not in features:
                continue
            group = features[col].groupby(race_ids)
            features[col + '_relative'] = features[col] - group.transform('mean')
            features[col + '_race_rank'] = group.rank(pct=True, method='average')
        return features

    def update(self, completed):
        # Compute race-winner time only AFTER settlement of this historical date.
        # A timestamp from the future race is not referenced by transform().
        settled = completed.copy()
        winning_seconds = settled['_past_time_source'].where(
            settled['finishing_position_num'].eq(1))
        settled['_winner_time_prior_race'] = winning_seconds.groupby(settled['race_id']).transform('min')
        for row in settled.sort_values(['race_date', 'race_no', 'horse_number']).to_dict('records'):
            finish = row['finishing_position_num']
            top3 = int(finish <= 3)
            history = self.horses.setdefault(row['horse_id'], [])
            raw_odds = row.get('_lagged_win_odds_source', np.nan)
            win_odds = float(raw_odds) if pd.notna(raw_odds) and float(raw_odds) > 0 else np.nan
            return_value = (float(row['return_target']) if row.get('audit_ok', False)
                            and pd.notna(row.get('return_target', np.nan)) else np.nan)
            prior_field_denominator = max(1, row['field_size'] - 1)
            def frac_position(col):
                raw = row.get(col, np.nan)
                return ((float(raw)-1) / prior_field_denominator
                        if pd.notna(raw) and 1 <= float(raw) <= row['field_size'] else np.nan)
            early = frac_position('_past_first_call_source')
            late = frac_position('_past_last_call_source')
            avg_run = frac_position('_past_avg_running_source')
            finish_fraction = (min(finish, row['field_size']) - 1) / prior_field_denominator
            margin = row.get('_past_margin_source', np.nan)
            run_time = row.get('_past_time_source', np.nan)
            winner_time = row.get('_winner_time_prior_race', np.nan)
            time_margin = (max(0., float(run_time) - float(winner_time)) /
                           (row['race_distance'] / 1000.)
                           if pd.notna(run_time) and pd.notna(winner_time)
                           and pd.notna(row['race_distance']) and row['race_distance'] > 0
                           and 0 <= run_time - winner_time <= 15 else np.nan)
            history.append({
                'date': row['race_date'], 'top3': top3, 'won': int(finish == 1),
                'early_fraction': early, 'late_fraction': late,
                'avg_running_fraction': avg_run,
                'finish_fraction': finish_fraction,
                'late_gain_fraction': early - finish_fraction if np.isfinite(early) else np.nan,
                'fast_finish': float(early - finish_fraction > 0.25) if np.isfinite(early) else np.nan,
                'front_runner': float(early < 0.25) if np.isfinite(early) else np.nan,
                'time_margin_per_km': time_margin, 'margin_lengths': margin,
                'bodyweight': row['declared_horse_weight_num'],
                'carried_weight': row['actual_weight_num'],
                'class_level': row['race_class_num'], 'track': row['track'],
                'gross_return': return_value,
                'score': 1 - (min(finish, row['field_size']) - 1) / max(1, row['field_size'] - 1),
                'distance': row['race_distance'], 'racecourse': row['racecourse'],
                'weight': row['actual_weight_num'], 'win_odds': win_odds,
            })
            for role, key in self.group_keys(row):
                successes, runs = self.groups.get((role, key), (0, 0))
                self.groups[(role, key)] = (successes + top3, runs + 1)
                if np.isfinite(return_value):
                    total, n = self.value_groups.get((role, key), (0.0, 0))
                    self.value_groups[(role, key)] = (total + return_value, n + 1)
        if len(completed):
            self.last_date = completed['race_date'].max()

    def fit_transform(self, completed):
        pieces = []
        for _, day in completed.groupby('race_date', sort=True):
            pieces.append(self.transform(day))
            self.update(day)
        return pd.concat(pieces).reindex(completed['row_id'])


V6PastOnlyFeatures = PastOnlyFeatures

class V7PastOnlyFeatures(V6PastOnlyFeatures):
    """Additional pre-race context; outcome sources are read only in update()."""
    def __init__(self):
        super().__init__()
        self.context_horses = {}
        self.speed_pars = {}
        self.recent_connections = {}

    @staticmethod
    def surface(row):
        return 'AWT' if 'ALL WEATHER' in str(row.get('track', '')).upper() else 'TURF'

    def transform(self, runners):
        base = super().transform(runners)
        records = []
        connection_cache = {}
        for row in runners.to_dict('records'):
            h = self.context_horses.get(row['horse_id'], [])
            recent = h[-5:]
            surface = self.surface(row)
            same = [r for r in h if r['surface'] == surface]
            f = {'row_id': row['row_id'], 'f_surface': surface,
                 'f_surface_runs': len(same),
                 'f_surface_form5': self.avg(same[-5:], 'score'),
                 'f_surface_top3_bayes': (sum(r['top3'] for r in same)+2.5)/(len(same)+10),
                 'f_surface_switch': float(h[-1]['surface'] != surface) if h else np.nan,
                 'f_jockey_switch': float(h[-1]['jockey'] != row['jockey']) if h else np.nan,
                 'f_trainer_switch': float(h[-1]['trainer'] != row['trainer']) if h else np.nan,
                 'f_starts_last60d': sum((row['race_date']-r['date']).days <= 60 for r in h),
                 'f_history_new_horse': float(not h)}
            for key in ['market_share', 'market_rank_score', 'beat_market',
                        'penultimate_position', 'closing_gain', 'speed_vs_prior_par']:
                f['f_'+key+'_last5'] = self.avg(recent, key)
            f['f_previous_market_share'] = h[-1]['market_share'] if h else np.nan
            for role in ['jockey', 'trainer']:
                cache_key = (role, row[role], row['race_date'])
                if cache_key not in connection_cache:
                    history = self.recent_connections.get((role, row[role]), [])
                    history = [r for r in history if 0 < row['race_date'].toordinal()-r[0] <= 180]
                    connection_cache[cache_key] = (sum(r[1] for r in history)+2.5)/(len(history)+10)
                f['f_'+role+'_recent180_rate'] = connection_cache[cache_key]
            records.append(f)
        extra = pd.DataFrame(records).set_index('row_id')
        result = pd.concat([base, extra], axis=1)
        race = runners.set_index('row_id')['race_id'].reindex(result.index)
        front = result['f_prior_front_running_rate']
        known = front.notna().groupby(race).transform('sum')-front.notna().astype(int)
        others = front.fillna(0).groupby(race).transform('sum')-front.fillna(0)
        result['f_opponents_front_running_rate'] = others/known.replace(0, np.nan)
        result['f_pace_pressure_interaction'] = front*result['f_opponents_front_running_rate']
        for col in ['f_market_share_last5', 'f_speed_vs_prior_par_last5',
                    'f_surface_form5', 'f_closing_gain_last5']:
            result[col+'_relative'] = result[col]-result[col].groupby(race).transform('mean')
        return result

    def update(self, completed):
        for date, day in completed.groupby('race_date', sort=True):
            if self.last_date is not None and date <= self.last_date:
                raise ValueError('History updates must advance by complete dates.')
            d = day.sort_values(['race_no', 'horse_number']).copy()
            odds = d['_lagged_win_odds_source'].where(d['_lagged_win_odds_source'].ge(1))
            inv = 1/odds
            coverage = inv.notna().groupby(d['race_id']).transform('mean')
            d['_market_share'] = (inv/inv.groupby(d['race_id']).transform('sum')).where(coverage.ge(.90))
            rank = odds.groupby(d['race_id']).rank(method='average')
            d['_market_score'] = 1-(rank-1)/(d['field_size']-1).clip(lower=1)
            par_updates = {}
            for row in d.to_dict('records'):
                h = self.context_horses.setdefault(row['horse_id'], [])
                surface = self.surface(row)
                key = (row['racecourse'], surface, row['race_distance'])
                total, count = self.speed_pars.get(key, (0., 0))
                seconds = row['_past_time_source']
                speed = (seconds/(row['race_distance']/1000.)
                         if pd.notna(seconds) and row['race_distance'] > 0 else np.nan)
                figure = total/count-speed if count >= 30 and pd.notna(speed) else np.nan
                if pd.notna(speed):
                    a, n = par_updates.get(key, (0., 0))
                    par_updates[key] = (a+speed, n+1)
                positions = [row.get('running_position_'+str(i)+'_num', np.nan) for i in range(1, 7)]
                positions = [float(p) for p in positions if pd.notna(p) and 1 <= p <= row['field_size']]
                penultimate = (positions[-2]-1)/max(1, row['field_size']-1) if len(positions) >= 2 else np.nan
                finish = min(row['finishing_position_num'], row['field_size'])
                fraction = (finish-1)/max(1, row['field_size']-1)
                top3 = int(row['finishing_position_num'] <= 3)
                h.append(dict(date=date, surface=surface, jockey=row['jockey'], trainer=row['trainer'],
                              score=1-fraction, top3=top3, market_share=row['_market_share'],
                              market_rank_score=row['_market_score'],
                              beat_market=1-fraction-row['_market_score'],
                              penultimate_position=penultimate, closing_gain=penultimate-fraction,
                              speed_vs_prior_par=figure))
                for role in ['jockey', 'trainer']:
                    self.recent_connections.setdefault((role, row[role]), []).append((date.toordinal(), top3))
            for key, (a, n) in par_updates.items():
                total, count = self.speed_pars.get(key, (0., 0))
                self.speed_pars[key] = (total+a, count+n)
            super().update(day)



class V8PastOnlyFeatures(V7PastOnlyFeatures):
    """New features of *completed prior races*, not current-race payouts or odds.

    The observed PLACE outcome and prior win-market estimates are compared on the
    historical date when they occurred. The date-level updater advances *after*
    the date is scored, so future and same-day labels never enter a predictor.
    """
    def __init__(self):
        super().__init__()
        self.market_form = {}
        self.draw_market_bias = {}

    @staticmethod
    def draw_key(row):
        # Previous-market-adjusted PLACE residuals by course, distance and gate
        # tercile. A large pseudo-count is essential for sparse combinations.
        n = max(int(row['field_size']), 1)
        draw = row.get('draw_num', np.nan)
        if not np.isfinite(draw):
            return None
        zone = min(2, max(0, int(3 * (draw - 1) / n)))
        dist = int(row['race_distance']) if np.isfinite(row['race_distance']) else -1
        surf = V7PastOnlyFeatures.surface(row)
        return (row['racecourse'], surf, dist, zone)

    def transform(self, runners):
        base = super().transform(runners)
        extra = []
        for row in runners.to_dict('records'):
            h = self.market_form.get(row['horse_id'], [])
            last3, last5, last10 = h[-3:], h[-5:], h[-10:]
            current_class = row['race_class_num']
            rated_form = [v['place_surprise'] for v in h[-8:] if np.isfinite(v['place_surprise'])]
            f = {'row_id': row['row_id']}
            f['f_v8_market_surprise_last3'] = self.avg(last3, 'place_surprise')
            f['f_v8_market_surprise_last10'] = self.avg(last10, 'place_surprise')
            f['f_v8_market_surprise_smoothed'] = (
                float(sum(rated_form)) / (len(rated_form) + 8) if rated_form else 0.0)
            f['f_v8_prior_market_place_prob'] = self.avg(last5, 'expected_place')
            f['f_v8_prior_market_strength_var'] = float(np.std([
                x['expected_place'] for x in last10 if np.isfinite(x['expected_place'])
            ])) if any(np.isfinite(x['expected_place']) for x in last10) else np.nan
            f['f_v8_prior_market_rank_surprise'] = self.avg(last5, 'rank_surprise')
            f['f_v8_rest_since_second_last'] = (
                (h[-1]['date'] - h[-2]['date']).days if len(h) > 1 else np.nan)
            f['f_v8_form_std5'] = float(np.std([x['score'] for x in last5])) if last5 else np.nan
            f['f_v8_form_peak5'] = float(max(x['score'] for x in last5)) if last5 else np.nan
            f['f_v8_form_worst5'] = float(min(x['score'] for x in last5)) if last5 else np.nan
            f['f_v8_days_to_distance'] = (abs(row['race_distance']-h[-1]['distance'])
                if h and np.isfinite(row['race_distance']) and np.isfinite(h[-1]['distance']) else np.nan)
            f['f_v8_class_repeat'] = (float(current_class == h[-1]['class_level'])
                if h and np.isfinite(current_class) and np.isfinite(h[-1]['class_level']) else np.nan)
            # Avoid a spurious 1.0 or -1.0 from a tiny set of winners.
            key = self.draw_key(row)
            n, total = self.draw_market_bias.get(key, (0, 0.0)) if key is not None else (0, 0.0)
            f['f_v8_draw_bias_market_adjusted'] = total / (n + 120.0)
            f['f_v8_draw_bias_prior_sample'] = n
            # Opponent strengths use other horses' strictly historical estimates.
            extra.append(f)
        extra_df = pd.DataFrame(extra).set_index('row_id')
        result = pd.concat([base, extra_df], axis=1)
        race_ids = runners.set_index('row_id')['race_id'].reindex(result.index)
        # Unlike other HKJC datasets, the uploaded runner workbook has no
        # horse_rating or horse_age columns. Use past-only strength instead.
        for source_col, label in [('f_lagged_win_market_strength', 'market'),
                                  ('f_top3_recency5', 'recent_form')]:
            vals = result[source_col]
            result[f'f_v8_field_best_{label}'] = vals.groupby(race_ids).transform('max')
            result[f'f_v8_field_{label}_gap_to_best'] = result[f'f_v8_field_best_{label}']-vals
            result[f'f_v8_field_{label}_std'] = vals.groupby(race_ids).transform('std')
        for col in ['f_v8_market_surprise_last3', 'f_v8_market_surprise_smoothed',
                    'f_v8_prior_market_place_prob',
                    'f_v8_draw_bias_market_adjusted', 'f_v8_form_peak5']:
            val = result[col]
            result[col+'_vs_field'] = val-val.groupby(race_ids).transform('mean')
        return result

    def update(self, completed):
        # The whole date arrives only after its predictions have been snapshotted.
        # Use the win market of this COMPLETED race only; it never enters the
        # current race's features. Missing odds give missing surprise, not zero.
        for date, day in completed.groupby('race_date', sort=True):
            if self.last_date is not None and date <= self.last_date:
                raise ValueError('V8 history must advance by strictly increasing complete race dates.')
            d = day.sort_values(['race_no', 'horse_number']).copy()
            market_odds = d['_lagged_win_odds_source'].where(d['_lagged_win_odds_source'].ge(1))
            inverse = 1. / market_odds
            coverage = inverse.notna().groupby(d['race_id']).transform('mean')
            share = (inverse / inverse.groupby(d['race_id']).transform('sum')).where(coverage.ge(.90))
            expected_place = np.minimum(.98, 3.0 * share)
            for i, row in enumerate(d.to_dict('records')):
                p = expected_place.iloc[i]
                if not pd.notna(p):
                    p = np.nan
                actual = float(row['finishing_position_num'] <= 3)
                residual = actual - p if np.isfinite(p) else np.nan
                frac = ((min(row['finishing_position_num'],row['field_size'])-1) /
                    max(row['field_size']-1,1))
                # This continuous expectation is only a historical heuristic.
                surprise_rank = ((1-frac)-p) if np.isfinite(p) else np.nan
                self.market_form.setdefault(row['horse_id'], []).append({
                    'date': date, 'place_surprise': residual,
                    'expected_place': p, 'rank_surprise': surprise_rank,
                    'score': 1-frac, 'rating': row['horse_rating_num'],
                    'class_level': row['race_class_num'], 'distance': row['race_distance'],
                })
                key = self.draw_key(row)
                if key is not None and np.isfinite(residual):
                    n, total = self.draw_market_bias.get(key, (0,0.))
                    self.draw_market_bias[key] = (n+1, total+residual)
            super().update(day)

class V9PastOnlyFeatures(V8PastOnlyFeatures):
    """Pre-race tactical interactions plus lagged horse trends.

    ALL features use today's *declared* runner/track characteristics and previous
    completed races; neither the outcome nor today's odds are used.  The race
    aggregation observes the opponents' strictly historical metrics only.
    """
    def transform(self, runners):
        base = super().transform(runners)
        race_ids = runners.set_index('row_id')['race_id'].reindex(base.index)
        present = runners.set_index('row_id').reindex(base.index)
        prior_early = pd.to_numeric(base['f_prior_early_position'], errors='coerce')
        prior_form = pd.to_numeric(base['f_top3_recency5'], errors='coerce')
        prior_speed = pd.to_numeric(base['f_prior_time_margin_sec_per_km'], errors='coerce')
        draw = pd.to_numeric(base['f_draw_fraction'], errors='coerce')
        # Historical tactical speed: smaller early fraction = faster to the lead.
        # Unknown running styles remain missing (not incorrectly imputed to fast).
        fast = prior_early.lt(0.35).where(prior_early.notna()).astype(float)
        closer = prior_early.gt(0.65).where(prior_early.notna()).astype(float)
        fast_count = fast.fillna(0).groupby(race_ids).transform('sum')
        known_count = fast.notna().groupby(race_ids).transform('sum')
        draw_count = fast.where(draw.lt(.50), np.nan).fillna(0).groupby(race_ids).transform('sum')
        form_mean = prior_form.groupby(race_ids).transform('mean')
        form_std = prior_form.groupby(race_ids).transform('std').replace(0, np.nan)
        speed_mean = prior_speed.groupby(race_ids).transform('mean')
        carried_weight = pd.to_numeric(base['f_actual_weight_num'], errors='coerce')
        body_weight = pd.to_numeric(base['f_declared_horse_weight_num'], errors='coerce')
        prior_market = pd.to_numeric(base['f_lagged_win_market_strength'], errors='coerce')
        future_extra = pd.DataFrame({
            'f_v9_fast_horse_count': fast_count,
            'f_v9_known_pace_horses': known_count,
            'f_v9_fast_pace_fraction': fast_count / known_count.replace(0, np.nan),
            'f_v9_fast_runners_inside': draw_count,
            'f_v9_fast_opponents': fast_count-fast.fillna(0),
            'f_v9_lone_lead': ((fast_count <= 1) & fast.eq(1)).astype(float).where(fast.notna()),
            'f_v9_hot_pace_closer': closer * (fast_count / known_count.replace(0, np.nan)),
            'f_v9_early_pos_draw_interaction': (1-prior_early)*(1-draw),
            'f_v9_early_pos_vs_field': prior_early-prior_early.groupby(race_ids).transform('mean'),
            'f_v9_form_vs_field_z': ((prior_form-form_mean)/form_std).clip(-4,4),
            'f_v9_form_field_median_gap': prior_form-prior_form.groupby(race_ids).transform('median'),
            'f_v9_speed_vs_field': prior_speed-speed_mean,
            'f_v9_speed_best_gap': prior_speed-prior_speed.groupby(race_ids).transform('min'),
            'f_v9_market_prior_field_rank': prior_market.groupby(race_ids).rank(ascending=False, pct=True),
            'f_v9_weight_vs_field': carried_weight-carried_weight.groupby(race_ids).transform('mean'),
            'f_v9_bodyweight_vs_field': body_weight-body_weight.groupby(race_ids).transform('mean'),
            'f_v9_distance_jump_abs': pd.to_numeric(base['f_distance_change'],errors='coerce').abs(),
            'f_v9_top3_short_long_gap': pd.to_numeric(base['f_top3_recency3'],errors='coerce')-
                                         pd.to_numeric(base['f_top3_recency10'],errors='coerce'),
            'f_v9_speed_fast_vs_form': (1-prior_early)*(prior_form-form_mean),
            'f_v9_class_move_abs': pd.to_numeric(base['f_class_move'],errors='coerce').abs(),
            'f_v9_rest_log': np.log1p(pd.to_numeric(base['f_days_since_run'], errors='coerce').clip(lower=0)),
            'f_v9_market_surprise_form_interaction': (
                pd.to_numeric(base['f_v8_market_surprise_smoothed'],errors='coerce')*(prior_form-form_mean)),
        }, index=base.index)
        # Stable historical runner-to-connection continuity.  This is available
        # from the current declared jockey/trainer and *previous* completed races.
        continuity = []
        for row in present[['horse_id','jockey','trainer']].to_dict('records'):
            past = self.horses.get(str(row['horse_id']), [])
            last = past[-1] if past else None
            continuity.append({
                'f_v9_same_jockey_as_last': float(row['jockey'] == last.get('jockey')) if last else np.nan,
                'f_v9_same_trainer_as_last': float(row['trainer'] == last.get('trainer')) if last else np.nan,
            })
        future_extra = pd.concat([future_extra, pd.DataFrame(continuity, index=base.index)],axis=1)
        return pd.concat([base, future_extra],axis=1)

    def update(self, completed):
        # Add connection history only after the whole completed date is processed.
        super().update(completed)
        for row in completed.to_dict('records'):
            horse = self.horses.get(row['horse_id'], [])
            if horse:
                horse[-1]['jockey'] = row['jockey']
                horse[-1]['trainer'] = row['trainer']


class V9LitePastOnlyFeatures(V9PastOnlyFeatures):
    """Controlled V9 ablation: V7 core + tactical features, not all V8 inputs.

    An early development-only chronological classification check found that
    using every V8 market feature together with tactical features was noisier.
    We compute V8 features internally only for strictly prior state, then drop
    their raw inputs from the model matrix.  The retained V9 market surprise
    interaction is itself based only on previous completed races.
    """
    def transform(self, runners):
        result = super().transform(runners)
        return result.loc[:, ~result.columns.str.startswith('f_v8_')]




class V10ConnectionFeatures(V9LitePastOnlyFeatures):
    """Market-adjusted jockey/trainer historical effects, strictly previous dates.

    A historical win-odds share is a noisy surrogate for the market expectation
    of that horse placing, NOT an executable quoted PLACE price. We use only
    *previous completed* races and heavy smoothing. No current odds are read.
    """
    def __init__(self):
        super().__init__()
        self.connection_residual = {}

    @staticmethod
    def _keys(row):
        return [('jockey', str(row['jockey'])),
                ('trainer', str(row['trainer'])),
                ('jockey_trainer', str(row['jockey']) + '||' + str(row['trainer'])),
                ('horse_trainer', str(row['horse_id']) + '||' + str(row['trainer']))]

    def transform(self, runners):
        base = super().transform(runners)
        now = runners.set_index('row_id').reindex(base.index)
        extra = []
        for row in now.to_dict('records'):
            rec = {}
            for kind, key in self._keys(row):
                count, surprise = self.connection_residual.get((kind, key), (0, 0.0))
                # The prior residual is zero under a well-informed market;
                # do not interpret 1/1 as a 100%-confidence improvement.
                rec[f'f_v10_{kind}_surprise'] = surprise / (count + 100.0)
                rec[f'f_v10_{kind}_sample_count'] = count
            extra.append(rec)
        return pd.concat([base, pd.DataFrame(extra, index=base.index)], axis=1)

    def update(self, completed):
        # Evaluate the historical market prediction only AFTER the day is done.
        # Use completed-day win odds solely to build features of *later* dates.
        for _, day in completed.groupby('race_date', sort=True):
            d = day.sort_values(['race_no','horse_number'])
            odds = pd.to_numeric(d['_lagged_win_odds_source'], errors='coerce')
            inv = (1.0 / odds).where(odds.ge(1))
            coverage = inv.notna().groupby(d['race_id']).transform('mean')
            share = inv / inv.groupby(d['race_id']).transform('sum')
            slots = np.where(d['field_size'] >= 7, 3.0, 2.0)
            expected_place = np.minimum(0.98, share * slots)
            expected_place = expected_place.where(coverage.ge(.90))
            for i, row in enumerate(d.to_dict('records')):
                p = expected_place.iloc[i]
                if not pd.notna(p):
                    continue
                # Smaller fields may have two paid places, rather than three.
                paid_places = 3 if row['field_size'] >= 7 else 2
                residual = float(row['finishing_position_num'] <= paid_places) - float(p)
                for key in self._keys(row):
                    n, s = self.connection_residual.get(key, (0, 0.0))
                    self.connection_residual[key] = n + 1, s + residual
        super().update(completed)


class V10CompactPastOnlyFeatures(V10ConnectionFeatures):
    """Small fixed predictor list; no data-driven feature selection from tests.

    Omitting hundreds of overlapping effects is intended to reduce variance.
    Every selected variable is pre-race or an earlier-date historical aggregate.
    """
    KEEP = {
        'f_race_course','f_race_class','f_track','f_track_condition',
        'f_actual_weight_num','f_declared_horse_weight_num','f_draw_fraction',
        'f_field_size','f_month_sin','f_month_cos','f_class_level',
        'f_horse_runs','f_days_since_run','f_horse_top3_bayes',
        'f_horse_wins_bayes','f_top3_last3','f_top3_last5','f_top3_last10',
        'f_score_recency3','f_score_recency5','f_score_volatility5',
        'f_prev_finish_score','f_weight_change','f_distance_change',
        'f_similar_distance_form','f_same_course_form','f_same_class_form',
        'f_distance_place_bayes','f_prior_early_position',
        'f_prior_late_gain','f_prior_time_margin_sec_per_km',
        'f_prior_length_margin','f_prior_fast_finishing_rate',
        'f_recent_speed_improvement','f_bodyweight_change_relative',
        'f_carried_weight_change','f_class_move','f_same_track_form',
        'f_v9_fast_opponents','f_v9_lone_lead',
        'f_v9_early_pos_draw_interaction','f_v9_form_vs_field_z',
        'f_v9_speed_vs_field','f_v9_weight_vs_field',
        'f_v9_rest_log','f_v9_same_jockey_as_last','f_v9_same_trainer_as_last',
        'f_v10_jockey_surprise','f_v10_jockey_sample_count',
        'f_v10_trainer_surprise','f_v10_trainer_sample_count',
        'f_v10_jockey_trainer_surprise','f_v10_jockey_trainer_sample_count',
        'f_v10_horse_trainer_surprise','f_v10_horse_trainer_sample_count',
    }
    def transform(self, runners):
        all_features = super().transform(runners)
        return all_features.loc[:, [c for c in all_features if c in self.KEEP]]


def build_block_features(history, future, feature_class, history_mode='frozen', outcome_loader=None):
    """Optional daily history advances after each complete date's feature snapshot.
    Weights, encoders and policy remain fixed. Frozen mode reads no future outcomes.
    """
    if history_mode not in {'frozen', 'daily'}:
        raise ValueError('history_mode must be frozen or daily')
    dates = np.sort(history['race_date'].unique())
    if len(dates) < MIN_FIT_DATES+MIN_CALIBRATION_DATES:
        raise ValueError('Too few historical race dates to fit and calibrate.')
    if history['race_date'].max() >= future['race_date'].min():
        raise ValueError('History must strictly precede prediction block.')
    ncal = max(MIN_CALIBRATION_DATES, int(len(dates)*.15))
    cutoff = pd.Timestamp(dates[-ncal])
    fit = history.loc[history['race_date'] < cutoff].copy()
    calibration = history.loc[history['race_date'] >= cutoff].copy()
    state = feature_class()
    xf = state.fit_transform(fit)
    if history_mode == 'daily':
        xc = state.fit_transform(calibration)
        if outcome_loader is None:
            raise ValueError('Daily mode requires a date-scoped settlement callback.')
        parts = []
        for date, day in future.groupby('race_date', sort=True):
            parts.append(state.transform(day))
            settled = outcome_loader(day.copy())
            if (set(settled['row_id']) != set(day['row_id']) or
                    not settled['race_date'].eq(date).all()):
                raise ValueError('History callback must return exactly this completed date.')
            state.update(settled)
        xp = pd.concat(parts).reindex(future['row_id'])
    else:
        xc = state.transform(calibration)
        state.update(calibration)
        xp = state.transform(future)
    return fit, calibration, xf, xc, xp




def build_shared_feature_cache(history, future, requested_sets, outcome_loader=None):
    """Compute V7/V8/V9-Lite/V10 historical states ONCE instead of four times.

    The feature classes form a superclass chain. V10's chronological history
    update invokes all parent updates. Intercept intermediate transform results
    without using any current-race outcome. Date ordering is unchanged.
    The resulting model matrices match independent feature generation.
    """
    allowed = {'v7', 'v8', 'v9_lite', 'v10_compact'}
    if not set(requested_sets).issubset(allowed):
        raise ValueError('Shared history optimization supports only V7/V8/V9-Lite/V10.')
    captures = {name: [] for name in allowed}
    existing = {
        'v7': (V7PastOnlyFeatures, V7PastOnlyFeatures.transform),
        'v8': (V8PastOnlyFeatures, V8PastOnlyFeatures.transform),
        'v9_lite': (V9LitePastOnlyFeatures, V9LitePastOnlyFeatures.transform),
        'v10_compact': (V10CompactPastOnlyFeatures, V10CompactPastOnlyFeatures.transform),
    }
    for name, (cls, func) in existing.items():
        def wrapper(self, runners, _func=func, _name=name):
            values = _func(self, runners)
            # No mutation of features, and no outcome data read by transform.
            if getattr(self, '_collect_intermediate_features', False):
                captures[_name].append(values)
            return values
        cls.transform = wrapper
    try:
        class SharedFeatureState(V10CompactPastOnlyFeatures):
            def __init__(self):
                super().__init__()
                self._collect_intermediate_features = True
        # This performs one forward chronological pass across all four levels.
        fit, calibration, _, _, _ = build_block_features(
            history, future, SharedFeatureState, HISTORY_MODE, outcome_loader)
    finally:
        # Never leave monkey-patched methods behind if something raises.
        for cls, func in existing.values():
            cls.transform = func
    output = {}
    # Each date is transformed only once before its results are learned.
    for name in requested_sets:
        full = pd.concat(captures[name], axis=0)
        if full.index.has_duplicates:
            raise AssertionError(f'Duplicate row ids in shared {name} features')
        output[name] = (full.reindex(fit['row_id']),
                        full.reindex(calibration['row_id']),
                        full.reindex(future['row_id']))
    return output

class ReturnReliabilityCalibrator:
    """Five calibration-only return bins, shrunk then monotonically pooled.
    Competes with legacy forecasts in discovery; no unsupported extrapolation.
    """
    def fit(self, predicted, actual):
        from sklearn.isotonic import IsotonicRegression
        p, y = np.asarray(predicted, float), np.asarray(actual, float)
        finite = np.isfinite(p) & np.isfinite(y)
        p, y = p[finite], y[finite]
        self.base = float(np.mean(y))
        edges = np.unique(np.quantile(p, np.linspace(0, 1, 6)))
        bucket = np.searchsorted(edges[1:-1], p, side='right')
        records = []
        for i in np.unique(bucket):
            mask = bucket == i
            n = int(mask.sum())
            mean = float(y[mask].mean())
            records.append(dict(predicted=float(p[mask].mean()), observed=mean, count=n,
                                shrunk=(n*mean+200*self.base)/(n+200)))
        self.table = records
        self.model = None
        if len(records) >= 2:
            self.model = IsotonicRegression(increasing=True, out_of_bounds='clip', y_min=0).fit(
                [r['predicted'] for r in records], [r['shrunk'] for r in records],
                sample_weight=[r['count']+200 for r in records])
        return self

    def predict(self, predicted):
        return (self.model.predict(np.asarray(predicted, float)) if self.model is not None
                else np.full(len(predicted), self.base))


class TrainOnlyEncoder:
    """Category vocabulary/medians fit only on model-fitting rows."""
    def fit(self, x):
        # Discard constant / entirely missing fields, based ONLY on training rows.
        # This also removes phantom columns absent from the uploaded workbook.
        self.columns = [col for col in x.columns if x[col].nunique(dropna=False) > 1]
        self.dropped_columns = sorted(set(x.columns) - set(self.columns))
        self.categories = {}
        self.medians = {}
        for col in self.columns:
            if not pd.api.types.is_numeric_dtype(x[col]):
                self.categories[col] = {v: i for i, v in enumerate(x[col].value_counts().head(120).index)}
            else:
                median = x[col].replace([np.inf, -np.inf], np.nan).median()
                self.medians[col] = float(median) if pd.notna(median) else 0.0
        self.category_mask = [col in self.categories for col in self.columns]
        return self

    def transform(self, x):
        # Column arrays avoid quadratic fragmentation from repeated DataFrame.insert.
        arrays = []
        for col in self.columns:
            if col in self.categories:
                series = x[col].map(self.categories[col]).astype(float)
            else:
                series = (pd.to_numeric(x[col], errors='coerce')
                          .replace([np.inf, -np.inf], np.nan)
                          .fillna(self.medians[col]))
            arrays.append(series.to_numpy(dtype=float))
        return np.column_stack(arrays)


def logit(values):
    p = np.clip(values, 1e-5, 1 - 1e-5)
    return np.log(p / (1 - p)).reshape(-1, 1)


def project_place_probabilities(probabilities, race_ids, field_sizes):
    """Race-coherent PLACE probabilities, using ONLY prediction-time inputs.

    Expected paid places: 3 for 7+ actual runners; leave smaller fields untouched.
    Apply a multiplicative capped normalization per race so p_i sums to
    the approximate number of paid places. If the field is smaller than
    4, retain calibrated probabilities (normally not a PLACE betting pool).
    This is a modeling assumption, not the post-race settlement count.
    """
    probs = np.clip(np.asarray(probabilities, dtype=float), 1e-5, 1.0-1e-5)
    race_ids = np.asarray(race_ids)
    field_sizes = np.asarray(field_sizes)
    result = probs.copy()
    for race in pd.unique(race_ids):
        mask = race_ids == race
        size = int(np.nanmedian(field_sizes[mask]))
        if size < 7:
            continue
        slots = 3
        if slots >= mask.sum():
            continue
        p = probs[mask]
        # Bisection for sum(min(p * scale, 0.999)) = slots.
        lo, hi = 0.0, 1.0
        while np.minimum(p * hi, 0.999).sum() < slots:
            hi *= 2.0
        for _ in range(40):
            mid = (lo + hi) / 2.0
            if np.minimum(p * mid, 0.999).sum() < slots:
                lo = mid
            else:
                hi = mid
        result[mask] = np.clip(p * hi, 1e-5, 0.999)
    return result


def shrunk_return_scale(actual, predicted, prior_strength=160):
    """Conservative, calibration-only adjustment toward scale=1.

    Do not calibrate with any outcomes from the scored future block.
    """
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    if len(actual) != len(predicted) or not len(actual):
        return 1.0
    ratio = np.mean(actual) / max(np.mean(predicted), 1e-8)
    n = len(actual)
    return float(np.clip((n * ratio + prior_strength) / (n + prior_strength), 0.75, 1.35))


def fit_predict_block(history, future, profiles=None, outcome_loader=None):
    """Fit on earlier dates, calibrate on later dates; score strictly future races.

    Only historical audited payouts train payout models. Current-race closing odds,
    final position and dividends are NEVER current-race features. Daily mode may
    use completed earlier evaluation dates as history; fitting and tuning stay prior.
    """
    _v13_block_start = time.perf_counter()
    profiles = MODEL_PROFILES if profiles is None else profiles
    dates = np.sort(history['race_date'].unique())
    if len(dates) < MIN_FIT_DATES + MIN_CALIBRATION_DATES:
        raise ValueError('Too few historical race dates to fit and calibrate.')
    ncal = max(MIN_CALIBRATION_DATES, int(len(dates) * 0.15))
    cutoff = pd.Timestamp(dates[-ncal])
    fit = history.loc[history['race_date'] < cutoff].copy()
    calibration = history.loc[history['race_date'] >= cutoff].copy()
    if history['race_date'].max() >= future['race_date'].min():
        raise ValueError('Training/calibration must precede the prediction block.')
    fit_safe = fit.loc[fit['audit_ok']].set_index('row_id')
    cal_safe = calibration.loc[calibration['audit_ok']].set_index('row_id')
    if (min(len(fit_safe), len(cal_safe)) < 20 or
            fit_safe['place_hit'].nunique() != 2 or cal_safe['place_hit'].nunique() != 2):
        raise ValueError('Need both classes and enough audited fit/calibration rows.')
    feature_cache, model_audits = {}, {}
    age = (fit_safe['race_date'].max() - fit_safe['race_date']).dt.days.to_numpy()
    weights = np.power(0.5, age / RECENCY_HALF_LIFE_DAYS)
    winners = fit_safe['place_hit'].eq(1).to_numpy()
    cal_winners = cal_safe['place_hit'].eq(1).to_numpy()
    if not winners.any() or not cal_winners.any():
        raise ValueError('Conditional return estimation requires historical winners.')
    result = future[IDENTIFIERS].copy().reset_index(drop=True)
    # Train-only upper bound stabilizes regression against one giant dividend.
    payout_cap = float(max(2.0, np.quantile(fit_safe['return_target'], 0.995)))
    capped_payout = fit_safe['return_target'].clip(upper=payout_cap).to_numpy()
    required_feature_sets = {p.get('feature_set', 'v7') for p in profiles.values()}
    if (SHARE_HISTORY_FEATURE_PASSES and len(required_feature_sets) >= 2
            and required_feature_sets <= {'v7', 'v8', 'v9_lite', 'v10_compact'}):
        t0 = time.perf_counter()
        feature_cache = build_shared_feature_cache(
            history, future, required_feature_sets, outcome_loader)
        print(f'  Shared history feature pass: {time.perf_counter()-t0:.1f}s '
              f'({len(required_feature_sets)} feature sets, 1 chronological scan)', flush=True)
    for name, profile in profiles.items():
        feature_set = profile.get('feature_set', 'v7')
        feature_class = {'v6': V6PastOnlyFeatures, 'v7': V7PastOnlyFeatures, 'v8': V8PastOnlyFeatures, 'v9': V9PastOnlyFeatures, 'v9_lite': V9LitePastOnlyFeatures,
                         'v10_compact': V10CompactPastOnlyFeatures}[feature_set]
        if feature_set not in feature_cache:
            _, _, xx, xc_all, xp_all = build_block_features(
                history, future, feature_class, HISTORY_MODE, outcome_loader)
            feature_cache[feature_set] = (xx, xc_all, xp_all)
        x_fit_all, x_cal_all, x_future = feature_cache[feature_set]
        encoder = TrainOnlyEncoder().fit(x_fit_all.loc[fit_safe.index])
        xf = encoder.transform(x_fit_all.loc[fit_safe.index])
        xc = encoder.transform(x_cal_all.loc[cal_safe.index])
        xp = encoder.transform(x_future)
        # Per-profile temporal decay. All settings are fixed before discovery.
        weights = np.power(0.5, age / profile.get('half_life_days', RECENCY_HALF_LIFE_DAYS))
        engine = profile.get('engine', 'histgb')
        if engine == 'histgb':
            common = dict(max_iter=BOOSTING_ITERATIONS, learning_rate=0.05, min_samples_leaf=40,
                          l2_regularization=10.0, early_stopping=False,
                          categorical_features=encoder.category_mask, random_state=42,
                          max_leaf_nodes=profile['max_leaf_nodes'])
            classifier = HistGradientBoostingClassifier(**common).fit(
                xf, fit_safe['place_hit'], sample_weight=weights)
            raw_cal = classifier.predict_proba(xc)[:, 1]
            raw_future = classifier.predict_proba(xp)[:, 1]
            direct = HistGradientBoostingRegressor(loss='poisson', **common).fit(
                xf, capped_payout, sample_weight=weights)
            conditional = HistGradientBoostingRegressor(loss='poisson', **common).fit(
                xf[winners], capped_payout[winners], sample_weight=weights[winners])
            dc, dp = direct.predict(xc), direct.predict(xp)
            cc, cp = conditional.predict(xc), conditional.predict(xp)
        elif engine == 'lightgbm':
            # Adds a materially different tree learner, using the exact same
            # train-only encoder, time weights, and calibration cutoffs.
            # Keep trees small to reduce overfitting on infrequent PLACE payouts.
            common = dict(n_estimators=profile['iterations'], num_leaves=profile['num_leaves'],
                          max_depth=5, min_child_samples=65, learning_rate=0.035,
                          colsample_bytree=0.85, reg_lambda=12., reg_alpha=1.,
                          random_state=42, n_jobs=2, verbosity=-1)
            lf = pd.DataFrame(xf, columns=encoder.columns)
            lc = pd.DataFrame(xc, columns=encoder.columns)
            lp = pd.DataFrame(xp, columns=encoder.columns)
            for column in encoder.categories:
                levels = list(encoder.categories[column].values())
                for frame in [lf, lc, lp]:
                    frame[column] = pd.Categorical(frame[column], categories=levels)
            classifier = LGBMClassifier(objective='binary', **common).fit(
                lf, fit_safe['place_hit'].astype(int), sample_weight=weights)
            raw_cal = classifier.predict_proba(lc)[:, 1]
            raw_future = classifier.predict_proba(lp)[:, 1]
            direct = LGBMRegressor(objective='poisson', **common).fit(
                lf, capped_payout, sample_weight=weights)
            conditional = LGBMRegressor(objective='poisson', **common).fit(
                lf.iloc[np.flatnonzero(winners)], capped_payout[winners], sample_weight=weights[winners])
            dc, dp = direct.predict(lc), direct.predict(lp)
            cc, cp = conditional.predict(lc), conditional.predict(lp)
        elif engine == 'catboost':
            if not HAS_CATBOOST:
                raise ImportError('CatBoost model selected, but catboost is not installed.')
            categorical_cols = list(encoder.categories)
            # These category encodings and medians are learned ONLY from the fit block.
            def to_catboost_frame(features):
                # Build all columns in one operation. Repeated insertion (V2) caused
                # the pandas DataFrame-is-highly-fragmented warnings in the log.
                cols = {}
                for col in encoder.columns:
                    if col in encoder.categories:
                        known = encoder.categories[col]  # membership lookup, fit-only vocabulary
                        vals = features[col].fillna('missing').astype(str)
                        cols[col] = vals.where(vals.isin(known), '__unseen__')
                    else:
                        cols[col] = (pd.to_numeric(features[col], errors='coerce')
                                     .replace([np.inf, -np.inf], np.nan)
                                     .fillna(encoder.medians[col]).astype(float))
                return pd.DataFrame(cols, index=features.index).copy()
            xcf = to_catboost_frame(x_fit_all.loc[fit_safe.index])
            xcc = to_catboost_frame(x_cal_all.loc[cal_safe.index])
            xcp = to_catboost_frame(x_future)
            kwargs = dict(iterations=profile['iterations'], depth=profile['depth'],
                          learning_rate=0.045, l2_leaf_reg=7.0, random_seed=42,
                          thread_count=2, allow_writing_files=False, verbose=False)
            classifier = CatBoostClassifier(loss_function='Logloss', **kwargs)
            classifier.fit(xcf, fit_safe['place_hit'], cat_features=categorical_cols,
                           sample_weight=weights)
            raw_cal = classifier.predict_proba(xcc)[:, 1]
            raw_future = classifier.predict_proba(xcp)[:, 1]
            # A log-RMSE target followed by expm1 estimates something closer
            # to the conditional MEDIAN, underestimating mean payouts (Jensen).
            # Poisson deviance uses a log-link to estimate the conditional MEAN
            # of nonnegative payout multiples, even though they are continuous.
            direct = CatBoostRegressor(loss_function='Poisson', **kwargs)
            direct.fit(xcf, capped_payout, cat_features=categorical_cols,
                       sample_weight=weights)
            conditional = CatBoostRegressor(loss_function='Poisson', **kwargs)
            conditional.fit(xcf.iloc[np.flatnonzero(winners)], capped_payout[winners],
                            cat_features=categorical_cols, sample_weight=weights[winners])
            dc = np.maximum(0, direct.predict(xcc))
            dp = np.maximum(0, direct.predict(xcp))
            cc = np.maximum(0, conditional.predict(xcc))
            cp = np.maximum(0, conditional.predict(xcp))
        else:
            raise ValueError(f'Unknown engine: {engine}')
        # Calibration uses later historical labels, never labels in future block.
        calibrator = LogisticRegression(C=1.0, random_state=42).fit(logit(raw_cal), cal_safe['place_hit'])
        p_raw = calibrator.predict_proba(logit(raw_future))[:, 1]
        p = project_place_probabilities(p_raw, future['race_id'], future['field_size'])
        direct_scale = shrunk_return_scale(cal_safe['return_target'], dc)
        conditional_scale = shrunk_return_scale(cal_safe.loc[cal_winners, 'return_target'], cc[cal_winners])
        # No extreme extrapolation beyond the fit-only payout cap.
        a = np.maximum(0, np.minimum(dp, payout_cap) * direct_scale)
        b = np.maximum(0, p * np.minimum(cp, payout_cap) * conditional_scale)
        pc = project_place_probabilities(
            calibrator.predict_proba(logit(raw_cal))[:, 1],
            cal_safe['race_id'], cal_safe['field_size'])
        cal_blend = (np.maximum(0, np.minimum(dc, payout_cap)*direct_scale) +
                     pc*np.maximum(0, np.minimum(cc, payout_cap)*conditional_scale))/2
        reliability = ReturnReliabilityCalibrator().fit(cal_blend, cal_safe['return_target'])
        result[name + '_calibrated'] = reliability.predict((a+b)/2)
        model_audits[name] = {
            'feature_set': feature_set, 'features': encoder.columns,
            'dropped': encoder.dropped_columns, 'feature_count': len(encoder.columns),
            'return_calibration_bins': reliability.table,
            'direct_scale': direct_scale, 'conditional_scale': conditional_scale}
        if V14_VALUE_ENABLED and name == 'v8_context':
            values, value_audit = fit_value_logistic(
                xf, xc, xp, fit_safe['return_target'].to_numpy(),
                cal_safe['return_target'].to_numpy(), weights, encoder.category_mask)
            result[name + '_' + V14_SCORE] = values
            model_audits[name]['v14_value_model'] = value_audit
        result[name + '_p'] = p
        for kind, values in [('direct', a), ('factorized', b),
                             ('blend', (a + b) / 2), ('robust', np.minimum(a, b))]:
            result[name + '_' + kind] = np.maximum(0, values)
    # Median ensemble minimizes dependency on one model's payoff extrapolation;
    # it never sees labels from the prediction period.
    ensemble_members = [m for m in V9_ENSEMBLE_MEMBERS if m in profiles]
    if len(ensemble_members) > 1:
        for suffix in ['p', 'direct', 'factorized', 'blend', 'robust', 'calibrated']:
            columns = [f'{m}_{suffix}' for m in ensemble_members]
            result[f'ensemble_{suffix}'] = np.median(result[columns].to_numpy(), axis=1)
        result['ensemble_p'] = project_place_probabilities(
            result['ensemble_p'].to_numpy(), future['race_id'], future['field_size'])
        # Predictive agreement is a pre-race property, not a realized outcome.
        # The 25th percentile discounts speculative returns forecast by only
        # one unusually optimistic model. Compare in development folds only.
        blends = result[[f'{m}_blend' for m in ensemble_members]].to_numpy(dtype=float)
        result['ensemble_consensus'] = np.quantile(blends, 0.25, axis=1)
        result['ensemble_model_disagreement'] = np.std(blends, axis=1)
    # V11 hybrid: two deliberately different time horizons. The old ensemble
    # uses the original V9 members (730-day recency); the compact model uses
    # 365-day recency, with fewer features. All weights are FIXED before OOF.
    # This creates a challenger without refitting an extra set of classifiers.
    if 'ensemble_p' in result and 'v10_compact_p' in result:
        name = V11_HYBRID_NAME
        old_w, recent_w = V11_LEGACY_WEIGHT, V11_COMPACT_WEIGHT
        result[f'{name}_p'] = project_place_probabilities(
            old_w * result['ensemble_p'].to_numpy(float) +
            recent_w * result['v10_compact_p'].to_numpy(float),
            future['race_id'], future['field_size'])
        for suffix in ('direct', 'factorized', 'blend', 'robust', 'calibrated'):
            result[f'{name}_{suffix}'] = np.maximum(0.0,
                old_w * result[f'ensemble_{suffix}'].to_numpy(float) +
                recent_w * result[f'v10_compact_{suffix}'].to_numpy(float))
        # Conservative confidence score penalizes disagreement, instead of
        # rewarding exceptional payouts estimated by just one submodel.
        legacy = result['ensemble_blend'].to_numpy(float)
        recent = result['v10_compact_blend'].to_numpy(float)
        mix = result[f'{name}_blend'].to_numpy(float)
        disagreement = np.abs(legacy - recent)
        result[f'{name}_consensus'] = np.maximum(0.0, mix - 0.35 * disagreement)
        result[f'{name}_model_disagreement'] = disagreement
    all_features = sorted(set(c for x in model_audits.values() for c in x['features']))
    dropped = sorted(set(c for x in model_audits.values() for c in x['dropped'])-set(all_features))
    provenance = {'fit_end': str(fit['race_date'].max().date()),
                  'calibration_start': str(calibration['race_date'].min().date()),
                  'calibration_end': str(calibration['race_date'].max().date()),
                  'prediction_start': str(future['race_date'].min().date()),
                  'prediction_end': str(future['race_date'].max().date()),
                  'fit_rows': len(fit_safe), 'calibration_rows': len(cal_safe),
                  'features': all_features, 'dropped_noninformative_features': dropped, 'model_details': model_audits,
                  'feature_count': len(all_features), 'payout_cap_from_fit': payout_cap,
                  'historic_return_features': 'previous audited PLACE dividends only',
                  'historic_running_style_and_speed_features': 'post-race observations only from completed prior dates',
                  'v9_pace_context': 'only pre-race draw and historical early-running style of today declared runners',
                  'v9_catboost_payout_loss': 'Poisson conditional mean instead of log1p RMSE conditional median',
                  'v8_historical_market_surprise': 'only realized earlier dates with earlier-race odds, never current odds',
                  'test_history': HISTORY_MODE,
                  'history_rule': 'strict earlier dates only; same-day and future outcomes prohibited',
                  'market_features': 'lagged previous-race win odds only; current odds never read',
                  'place_probability_projection': 'race-size-only normalization of independently calibrated probabilities',
                  'v9_additions': 'lean V7 core plus pace pressure, lone leaders, speed-vs-opponents, draw-pacing, jockey/trainer continuity',
                   'v10_additions': 'only prior-date market-surprise for jockey/trainer connections, fixed compact candidate, 365-day decay; V9 ensemble fixed',
                  'v11_additions': 'fixed 65/35 V9-ensemble/V10-compact hybrid, disagreement-penalized consensus; no current odds; development-only monthly diagnostics',
                  'v13_parent_v12_additions': 'predeclared ST/HV course-only gate; robust OOF gates unchanged; daily out-of-time course diagnostics',
                  'v11_blend_weights': {'legacy': V11_LEGACY_WEIGHT, 'compact': V11_COMPACT_WEIGHT},
                  'v13_shared_history_optimization': 'single model-family chronological feature pass, no lookahead; all original V12 models retained',
                  'v8_additions': 'historic market surprise, opponent historic market strength, prior draw bias, form volatility' }
    print(f'  Fit/predict block: {time.perf_counter()-_v13_block_start:.1f}s; 'f'models={len(profiles)}, prediction runners={len(future)}', flush=True)
    return result, provenance


def make_bet_plan(predictions, policy, active=True):
    """Input contains predictions and identifiers, never realized payouts."""
    p_col = policy['model'] + '_p'
    score_col = policy['model'] + '_' + policy['score']
    if any(c not in predictions for c in [p_col, score_col]):
        raise ValueError('Policy needs prediction-only score columns, never outcome columns.')
    plan = predictions[IDENTIFIERS + [p_col, score_col]].copy()
    plan = plan.rename(columns={p_col: 'predicted_place_probability', score_col: 'predicted_return'})
    plan = plan.sort_values(['race_date', 'racecourse', 'race_no', 'predicted_return', 'horse_number'],
                            ascending=[True, True, True, False, True], kind='mergesort')
    # The course is on the race card BEFORE betting. Do not use result or payout
    # to choose the venue. The venue choice itself is learned only on discovery.
    venue_filter = policy.get('venue_filter', 'ALL')
    if venue_filter not in {'ALL', 'ST', 'HV'}:
        raise ValueError(f'Unsupported predeclared venue filter: {venue_filter}')
    eligible = (plan['predicted_place_probability'].ge(policy['min_probability'])
                & plan['predicted_return'].ge(policy['min_return']))
    if venue_filter != 'ALL':
        eligible &= plan['racecourse'].eq(venue_filter)
    rank = plan.loc[eligible].groupby('race_id').cumcount() + 1
    plan['buy_flag'] = False
    if active:
        plan.loc[rank.index, 'buy_flag'] = rank.le(policy['top_n'])
    # One fixed unit: quantity increases cannot manufacture an expected advantage.
    plan['ticket_units'] = plan['buy_flag'].astype(int)
    return plan.sort_values('row_id').reset_index(drop=True)


def settle_plan(plan, outcomes):
    details = plan.merge(outcomes[['row_id', 'audit_ok', 'place_hit', 'return_target']],
                         on='row_id', how='left', validate='one_to_one')
    if details['audit_ok'].isna().any():
        raise ValueError('Every prediction must have a settlement audit row.')
    details['stake'] = details['ticket_units'] * UNIT_STAKE
    # Conservative accounting: unresolved purchases cost their stake and receive
    # zero assumed payout. Also report audited-only ROI separately for diagnosis.
    details['payout'] = details['stake'] * details['return_target'].fillna(0)
    details['profit'] = details['payout'] - details['stake']
    details['hit_bought'] = details['buy_flag'] & details['place_hit'].eq(1)
    return details


def summarize_settlement(details):
    bought = details.loc[details['buy_flag']]
    audited = bought.loc[bought['audit_ok']]
    stake = float(bought['stake'].sum())
    payout = float(bought['payout'].sum())
    audited_stake = float(audited['stake'].sum())
    return {'bought_tickets': len(bought), 'bought_races': int(bought['race_id'].nunique()),
            'hit_tickets': int(bought['hit_bought'].sum()),
            'unresolved_bought_tickets': int((~bought['audit_ok']).sum()),
            'total_stake': stake, 'total_payout': payout, 'total_profit': payout - stake,
            'roi': payout / stake if stake else np.nan,
            'audited_roi': float(audited['payout'].sum()) / audited_stake if audited_stake else np.nan,
            'hit_rate': float(bought['hit_bought'].mean()) if len(bought) else np.nan}


def bootstrap_roi(details, samples=BOOTSTRAP_SAMPLES):
    """Resample whole calendar weeks, retaining within-week bet dependence.

    These intervals are descriptive, not a multiple-search-adjusted guarantee.
    """
    bought = details.loc[details['buy_flag']].copy()
    if bought.empty:
        return np.nan, np.nan, np.nan
    bought['week'] = pd.to_datetime(bought['race_date']).dt.to_period('W').astype(str)
    groups = bought.groupby('week')[['stake', 'payout']].sum().to_numpy()
    if len(groups) < 8:
        return np.nan, np.nan, np.nan
    rng = np.random.RandomState(42)
    indices = rng.randint(0, len(groups), size=(samples, len(groups)))
    totals = groups[indices].sum(axis=1)
    roi = totals[:, 1] / totals[:, 0]
    return tuple(float(v) for v in np.quantile(roi, [0.025, 0.10, 0.975]))



class V13PolicyCache:
    """Cache OOF sorting and outcome alignment; compute only purchased rows.

    Selection criteria and week-bootstrap are the same as V12. Training labels
    are NOT included in model features; this cache is for discovery settlement.
    """
    def __init__(self, oof, outcomes):
        oof = oof.reset_index(drop=True)
        if oof['row_id'].duplicated().any() or outcomes['row_id'].duplicated().any():
            raise ValueError('Duplicate row ids in out-of-fold predictions/outcomes.')
        data = oof.copy()
        target = outcomes.set_index('row_id')[['audit_ok', 'place_hit', 'return_target']]
        aligned = target.reindex(data['row_id']).reset_index(drop=True)
        if aligned['audit_ok'].isna().any():
            raise ValueError('Missing audit for a discovery prediction.')
        self.table = pd.concat([data, aligned], axis=1)
        self.sorted_indices = {}
        self.all_folds = sorted(data['fold_id'].unique())

    def settle(self, policy):
        df = self.table
        name = policy['model']
        p_col = name + '_p'
        score_col = name + '_' + policy['score']
        key = (p_col, score_col)
        if key not in self.sorted_indices:
            # Stable ordering and tie breaker EXACTLY as make_bet_plan.
            self.sorted_indices[key] = df.sort_values(
                ['race_date','racecourse','race_no',score_col,'horse_number'],
                ascending=[True,True,True,False,True],kind='mergesort').index
        ordered = df.loc[self.sorted_indices[key]]
        mask = (ordered[p_col].ge(policy['min_probability'])
                & ordered[score_col].ge(policy['min_return']))
        scope = policy.get('venue_filter','ALL')
        if scope != 'ALL':
            mask &= ordered['racecourse'].eq(scope)
        candidates = ordered.loc[mask]
        chosen = candidates.groupby('race_id', sort=False).head(policy['top_n']).copy()
        chosen['buy_flag'] = True
        chosen['ticket_units'] = 1
        chosen['stake'] = UNIT_STAKE
        chosen['payout'] = UNIT_STAKE * chosen['return_target'].fillna(0)
        chosen['profit'] = chosen['payout'] - chosen['stake']
        chosen['hit_bought'] = chosen['place_hit'].eq(1)
        return chosen

def _v13_validation_search(oof, outcomes):
    """Chronological OOF search; settlement is used ONLY in discovery here.

    V12 tests just two course-only subgroups for three preselected model families,
    with exactly the same minimum sample / fold / CI requirements as the baseline.
    Choosing a course on previously inspected test performance is forbidden.
    """
    records = []
    _v13_policy_cache = V13PolicyCache(oof, outcomes) if FAST_POLICY_SEARCH else None
    candidates = list(MODEL_PROFILES) + (['ensemble'] if len(MODEL_PROFILES) > 1 else [])
    if V11_HYBRID_NAME+'_p' in oof:
        candidates.append(V11_HYBRID_NAME)
    specifications = []
    for model in candidates:
        score_list = (['blend', 'calibrated', 'consensus'] if model in ('ensemble', V11_HYBRID_NAME)
                      else ['blend', 'calibrated'])
        for score, min_return, min_probability, top_n in itertools.product(
                score_list, RETURN_THRESHOLDS, MIN_PROBABILITIES, TOP_N_OPTIONS):
            specifications.append({'model': model, 'score': score,
                                   'min_return': float(min_return),
                                   'min_probability': float(min_probability),
                                   'top_n': int(top_n), 'venue_filter': 'ALL'})
    for model in V12_SCOPED_MODELS:
        if model not in candidates:
            continue
        scores = ('blend', 'consensus') if model == 'ensemble' else ('blend',)
        for venue_filter, score, min_return, min_probability, top_n in itertools.product(
                V12_SCOPED_VENUES, scores, V12_SCOPED_RETURN_THRESHOLDS,
                V12_SCOPED_MIN_PROBABILITIES, V12_SCOPED_TOP_N):
            specifications.append({'model': model, 'score': score,
                                   'min_return': float(min_return),
                                   'min_probability': float(min_probability),
                                   'top_n': int(top_n), 'venue_filter': venue_filter})
    if V14_VALUE_ENABLED and 'v8_context_' + V14_SCORE in oof:
        for min_return, min_probability, top_n in itertools.product(
                RETURN_THRESHOLDS, MIN_PROBABILITIES, TOP_N_OPTIONS):
            specifications.append({'model': 'v8_context', 'score': V14_SCORE,
                'min_return': float(min_return), 'min_probability': float(min_probability),
                'top_n': int(top_n), 'venue_filter': 'ALL'})
    for policy in specifications:
        if _v13_policy_cache is not None:
            detail = _v13_policy_cache.settle(policy)
            fold_stats = [summarize_settlement(detail.loc[detail['fold_id'] == fold])
                          for fold in _v13_policy_cache.all_folds]
        else:
            detail = settle_plan(make_bet_plan(oof, policy), outcomes)
            detail = detail.merge(oof[['row_id', 'fold_id']], on='row_id', validate='one_to_one')
            fold_stats = [summarize_settlement(g) for _, g in detail.groupby('fold_id')]
        stats = summarize_settlement(detail)
        lo, score_lcb, hi = bootstrap_roi(detail)
        enough = (stats['bought_tickets'] >= MIN_VALIDATION_BETS
                  and len(fold_stats) == oof['fold_id'].nunique()
                  and all(s['bought_tickets'] >= MIN_BETS_PER_FOLD for s in fold_stats))
        worst_roi = min((s['roi'] if np.isfinite(s['roi']) else -np.inf)
                        for s in fold_stats)
        eligible = bool(enough and np.isfinite(score_lcb) and score_lcb > 1.0
                        and worst_roi >= MIN_FOLD_ROI)
        reasons = []
        if stats['bought_tickets'] < MIN_VALIDATION_BETS:
            reasons.append('too_few_discovery_bets')
        if any(s['bought_tickets'] < MIN_BETS_PER_FOLD for s in fold_stats):
            reasons.append('too_few_bets_in_one_or_more_folds')
        if not (np.isfinite(score_lcb) and score_lcb > 1.0):
            reasons.append('discovery_lower90_at_or_below_one')
        if worst_roi < MIN_FOLD_ROI:
            reasons.append('worst_fold_roi_below_floor')
        records.append({**policy, **stats, 'bootstrap_low95': lo,
                        'selection_lower90': score_lcb, 'bootstrap_high95': hi,
                        'worst_fold_roi': worst_roi,
                        'fold_bet_counts': '/'.join(str(s['bought_tickets']) for s in fold_stats),
                        'fold_rois': '/'.join('nan' if not np.isfinite(s['roi'])
                                              else f"{s['roi']:.3f}" for s in fold_stats),
                        'rejection_reasons': ';'.join(reasons) if reasons else 'none',
                        'eligible': eligible})
    table = pd.DataFrame(records)
    def fold_min_count(value):
        return min(int(x) for x in value.split('/'))
    table['min_fold_bets'] = table['fold_bet_counts'].map(fold_min_count)
    table['sample_qualified'] = (table['bought_tickets'] >= MIN_VALIDATION_BETS) & (table['min_fold_bets'] >= MIN_BETS_PER_FOLD)
    table['support_fraction'] = np.minimum(table['bought_tickets']/MIN_VALIDATION_BETS, 1.0) * np.minimum(table['min_fold_bets']/MIN_BETS_PER_FOLD, 1.0)
    # Keep unscoped policies first if discovery evidence is tied: fewer degrees
    # of freedom reduces selection bias. Otherwise apply the predeclared gates.
    table['unscoped_preference'] = table['venue_filter'].eq('ALL').astype(int)
    # Validation is ~45% of development race dates; confirmation is ~15%.
    # Require a safety buffer for the 50-ticket confirmation requirement;
    # this is only a prioritization AMONG already eligible discovery policies.
    # No confirmation/test payouts or results are read to compute it.
    table['projected_confirmation_tickets'] = table['bought_tickets'] / 3.0
    table['confirmation_coverage_buffer'] = (
        table['projected_confirmation_tickets'] >= MIN_PROJECTED_CONFIRMATION_BETS
    ).astype(int)
    table = table.sort_values(
        ['eligible', 'confirmation_coverage_buffer', 'sample_qualified', 'support_fraction', 'selection_lower90',
         'worst_fold_roi', 'roi', 'unscoped_preference'],
        ascending=False, na_position='last').reset_index(drop=True)
    best = table.iloc[0]
    policy = {key: best[key] for key in ['model', 'score', 'min_return', 'min_probability', 'top_n', 'venue_filter']}
    policy['top_n'] = int(policy['top_n'])
    return policy, bool(best['eligible']), table


def development_probability_ablation(oof, outcome_rows, directory):
    """Chronological OOF comparison, never final test; does not choose a model."""
    models = [m for m in MODEL_PROFILES if f'{m}_p' in oof]
    if not models:
        return
    labels = outcome_rows[['row_id', 'place_hit', 'audit_ok', 'race_date']]
    temp = oof[['row_id','fold_id']+[m+'_p' for m in models]].merge(
        labels, on='row_id', how='left', validate='one_to_one')
    temp = temp.loc[temp['audit_ok'].eq(True) & temp['place_hit'].notna()].copy()
    if not len(temp):
        return
    records = []
    for fold, group in [('all',temp)]+[(str(k),v) for k,v in temp.groupby('fold_id')]:
        y = group['place_hit'].to_numpy(float)
        for model in models:
            prob = group[model+'_p'].to_numpy(float).clip(1e-6,1-1e-6)
            records.append({'fold':fold,'model':model,'runner_rows':len(group),
                'brier':float(np.mean((prob-y)**2)),
                'logloss':float(np.mean(-y*np.log(prob)-(1-y)*np.log(1-prob))),
                'average_probability':float(prob.mean()),'place_frequency':float(y.mean())})
    diag = pd.DataFrame(records)
    diag.to_csv(directory/'development_probability_ablation_v7_v8_v9_v10.csv',index=False)
    print('V7–V10 probability ablation (LOWER Brier is better; NOT ROI):')
    print(diag.loc[diag['fold'].eq('all')].round(5).to_string(index=False),flush=True)
    if 'v7_context' in models and 'v9_tactical' in models:
        y = temp['place_hit'].to_numpy(float)
        base = (temp['v7_context_p'].to_numpy(float)-y)**2
        tactical = (temp['v9_tactical_p'].to_numpy(float)-y)**2
        delta = pd.DataFrame({'date':temp['race_date'], 'delta':base-tactical}).groupby('date')['delta'].mean()
        print(f'V9 vs V7 Brier improvement (positive is better) averaged across prior dates: {delta.mean():.6f}')
    if 'v10_compact' in models:
        baseline = 'v7_context' if 'v7_context' in models else models[0]
        y = temp['place_hit'].to_numpy(dtype=float)
        ref_loss = (temp[baseline+'_p'].to_numpy(dtype=float)-y)**2
        new_loss = (temp['v10_compact_p'].to_numpy(dtype=float)-y)**2
        gain = pd.DataFrame({'date': temp['race_date'], 'brier_gain': ref_loss-new_loss})
        gain = gain.groupby('date')['brier_gain'].mean()
        print(f'V10 compact Brier gain over {baseline} on discovery dates '
              f'(positive better, not ROI): {gain.mean():.6f}', flush=True)



def save_v12_discovery_diagnostics(oof, outcomes, policy, directory):
    """Audit development predictions and realized returns. Never use test rows.

    Report calibration and temporal concentration independently of any score
    optimization. Reports are descriptive, not additional selection criteria.
    """
    forecast = settle_plan(make_bet_plan(oof, policy, active=True), outcomes)
    observed = forecast.loc[forecast['buy_flag'] & forecast['audit_ok'].eq(True)].copy()
    if observed.empty:
        print('V12 diagnostic: no audited development bets for selected policy')
        return
    observed['month'] = pd.to_datetime(observed['race_date']).dt.to_period('M').astype(str)
    grouped = observed.groupby('month', sort=True)
    monthly = grouped.agg(
        tickets=('stake','size'), stake=('stake','sum'),
        payout=('payout','sum'), observed_wins=('hit_bought','sum'),
        predicted_return=('predicted_return','mean'),
        realized_return=('return_target','mean')).reset_index()
    monthly['roi'] = monthly['payout'] / monthly['stake']
    monthly['profit'] = monthly['payout'] - monthly['stake']
    monthly.to_csv(directory / 'v12_development_monthly_roi.csv', index=False)
    observed['return_decile'] = pd.qcut(
        observed['predicted_return'].rank(method='first'),
        q=min(10, len(observed)), labels=False, duplicates='drop') + 1
    calibration = observed.groupby('return_decile', sort=True).agg(
        tickets=('stake','size'), predicted_return=('predicted_return','mean'),
        realized_return=('return_target','mean'),
        wins=('hit_bought','sum'), stake=('stake','sum'),
        payout=('payout','sum')).reset_index()
    calibration['roi'] = calibration['payout'] / calibration['stake']
    calibration.to_csv(directory / 'v12_development_return_deciles.csv', index=False)
    print('V12 selected policy development monthly stability:')
    print(monthly[['month','tickets','roi','profit']].to_string(index=False),flush=True)
    print('V12 predicted vs realized return deciles (development ONLY):')
    print(calibration[['return_decile','tickets','predicted_return',
                       'realized_return']].round(3).to_string(index=False), flush=True)
    # Avoid claiming a positive mean from a tiny isolated period.
    print(f'V12 diagnostic: {len(monthly)} development months, '
          f'{sum(monthly.roi > 1)} months over break-even; '
          'selection still uses the frozen discovery risk gates.', flush=True)


def write_json(path, value):
    def clean(obj):
        if isinstance(obj, dict):
            return {str(k): clean(v) for k, v in obj.items()}
        if isinstance(obj, (list, tuple)):
            return [clean(v) for v in obj]
        if isinstance(obj, (np.integer, np.floating, np.bool_)):
            obj = obj.item()
        if isinstance(obj, float) and not np.isfinite(obj):
            return None
        if obj is None or isinstance(obj, (str, int, float, bool)):
            return obj
        return str(obj)
    Path(path).write_text(json.dumps(clean(value), ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')


def plot_final_report(details, period, output_dir):
    fig, axes = plt.subplots(1, 3, figsize=(17, 4.6))
    bought = details.loc[details['buy_flag']].sort_values(['race_date', 'race_no'])
    hits = bought.loc[bought['hit_bought']]
    if len(bought):
        daily = bought.groupby('race_date')[['profit', 'stake', 'payout']].sum()
        axes[0].plot(daily.index, daily['profit'].cumsum(), color='#187C83')
        axes[1].plot(daily.index, daily['payout'].cumsum() / daily['stake'].cumsum(), color='#187C83')
        axes[2].scatter(hits['predicted_place_probability'], hits['return_target'], color='#187C83', alpha=0.6)
    else:
        for ax in axes:
            ax.text(0.5, 0.5, 'No bets: validation/confirmation gate or thresholds',
                    transform=ax.transAxes, ha='center', wrap=True, fontsize=9)
    axes[0].axhline(0, color='#B15E22', linestyle='--')
    axes[0].set_title('Cumulative flat-stake profit')
    axes[0].set_ylabel('Simulated stake currency')
    axes[1].axhline(1, color='#B15E22', linestyle='--')
    axes[1].set_title('Cumulative test ROI')
    axes[2].set_title('Purchased hits: prediction versus payout odds')
    axes[2].set_xlabel('Calibrated PLACE probability')
    axes[2].set_ylabel('Final gross payout / stake')
    axes[2].set_xlim(0, 1)
    axes[2].xaxis.set_major_formatter(PercentFormatter(1))
    for ax in axes:
        ax.grid(alpha=0.2)
    for ax in axes[:2]:
        ax.tick_params(axis='x', rotation=30)
    fig.suptitle(period + ' | locked strategy | final historical test', fontsize=14)
    fig.tight_layout()
    path = Path(output_dir) / 'final_test_graphs.png'
    fig.savefig(path, dpi=160, bbox_inches='tight')
    if SHOW_PLOTS:
        plt.show()
    plt.close(fig)
    return str(path)


def save_confirmation_diagnostics(predictions, outcomes, policy, bought_details, directory):
    """Diagnostic tables use confirmation labels, never the final-test block.

    The selected policy is already fixed by discovery before these are built.
    They are reports only, not additional selection rules.
    """
    model = policy['model']
    pcol, rcol = f"{model}_p", f"{model}_{policy['score']}"
    diag = predictions[['row_id', 'race_date', 'racecourse', pcol, rcol]].merge(
        outcomes[['row_id', 'race_class', 'race_distance', 'audit_ok',
                  'place_hit', 'return_target']], on='row_id', validate='one_to_one')
    diag = diag.loc[diag['audit_ok']].copy()
    # Calibration should be diagnosed over ALL valid runners, not just bets.
    diag['probability_bin'] = pd.cut(
        diag[pcol], bins=np.linspace(0, 1, 11), include_lowest=True)
    calibration = diag.groupby('probability_bin', observed=True).agg(
        runners=('row_id','size'), mean_pred_probability=(pcol,'mean'),
        observed_place_rate=('place_hit','mean'),
        mean_predicted_gross_return=(rcol,'mean'),
        observed_mean_gross_return=('return_target','mean')).reset_index()
    calibration['probability_bin'] = calibration['probability_bin'].astype(str)
    calibration.to_csv(directory / 'confirmation_probability_calibration.csv', index=False)
    print('Confirmation PLACE probability calibration (diagnostic only):')
    print(calibration[['probability_bin','runners','mean_pred_probability',
                       'observed_place_rate']].to_string(index=False), flush=True)
    bought = bought_details.loc[bought_details['buy_flag']].copy()
    bought = bought.merge(outcomes[['row_id','race_class','race_distance']],
                          on='row_id', validate='one_to_one')
    if len(bought):
        bought['distance_group'] = pd.cut(
            bought['race_distance'], bins=[0,1199,1399,1799,9999],
            labels=['sprint_lt1200','1200_1399','1400_1799','1800_plus'])
        summaries = []
        for name, col in [('racecourse','racecourse'),
                          ('race_class','race_class'), ('distance_group','distance_group')]:
            for key, part in bought.groupby(col, dropna=False, observed=True):
                st, pay = float(part['stake'].sum()), float(part['payout'].sum())
                summaries.append({'category':name,'value':str(key),
                                  'bets':len(part), 'stake':st,
                                  'payout':pay,'roi':pay/st if st else np.nan,
                                  'hits':int(part['hit_bought'].sum())})
        pd.DataFrame(summaries).to_csv(directory / 'confirmation_bet_subgroups.csv',index=False)
    return calibration


def save_dataset_audit(raw, directory):
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    direct = {'horse_number', 'horse_id', 'race_id', 'race_date_iso', 'race_course',
              'race_number', 'actual_weight_num', 'declared_horse_weight_num',
              'draw_num', 'race_distance', 'race_class', 'track_condition', 'track',
              'jockey', 'trainer', 'horse_rating', 'horse_age'}
    historical = {'finishing_position', 'finishing_position_num', 'win_odds', 'win_odds_num',
                  'first_call_position', 'last_call_position', 'avg_running_position',
                  'finish_time_seconds', 'length_behind_winner_num'}
    historical.update('running_position_'+str(i)+'_num' for i in range(1, 7))
    records = []
    for col in raw:
        role = ('pre_race_or_identifier' if col in direct else
                'past_completed_races_only' if col in historical else 'excluded')
        records.append(dict(column=col, dtype=str(raw[col].dtype),
                            missing_fraction=float(raw[col].isna().mean()),
                            unique_values=int(raw[col].nunique()), role=role))
    pd.DataFrame(records).to_csv(directory/'dataset_column_audit.csv', index=False)
    valid = raw[['last_call_position', 'finishing_position_num']].dropna()
    findings = {'rows': len(raw), 'columns': len(raw.columns),
                'last_call_equals_finish_fraction': float(valid.iloc[:, 0].eq(valid.iloc[:, 1]).mean()),
                'current_odds_used': False, 'current_sectional_times_used': False,
                'source_and_split_columns_used': False,
                'note': 'The result-file track condition/weights are assumed available at bet time; verify operational snapshots.'}
    write_json(directory/'dataset_findings.json', findings)
    print('Dataset audit:', findings, flush=True)


def save_prediction_diagnostics(predictions, outcomes, directory, stage):
    """Inspect all runners and return-score reliability; never retune on test."""
    data = predictions.merge(outcomes[['row_id', 'audit_ok', 'place_hit', 'return_target']],
                             on='row_id', validate='one_to_one')
    data = data.loc[data['audit_ok']].copy()
    models = [col[:-2] for col in predictions if col.endswith('_p')]
    metrics, reliability = [], []
    groups = [('all', data)]
    if 'fold_id' in data:
        groups += [(str(i), g) for i, g in data.groupby('fold_id')]
    for fold, group in groups:
        y = group['place_hit'].to_numpy()
        for model in models:
            p = group[model+'_p'].clip(1e-6, 1-1e-6).to_numpy()
            metrics.append(dict(stage=stage, fold=fold, model=model, rows=len(group),
                                brier=float(np.mean((p-y)**2)),
                                log_loss=float(-np.mean(y*np.log(p)+(1-y)*np.log(1-p))),
                                predicted_place_rate=float(p.mean()),
                                observed_place_rate=float(y.mean())))
            for score in ['blend', 'calibrated', 'consensus', V14_SCORE]:
                col = model+'_'+score
                if col not in group:
                    continue
                categories = pd.cut(group[col], [-np.inf,.8,1.,1.1,1.25,np.inf])
                for band, part in group.groupby(categories, observed=True):
                    reliability.append(dict(stage=stage, fold=fold, model=model, score=score,
                                            return_band=str(band), runners=len(part),
                                            predicted_return=float(part[col].mean()),
                                            realized_return=float(part['return_target'].mean()),
                                            hit_rate=float(part['place_hit'].mean()),
                                            hits=int(part['place_hit'].sum())))
    pd.DataFrame(metrics).to_csv(directory/(stage+'_model_metrics.csv'), index=False)
    pd.DataFrame(reliability).to_csv(directory/(stage+'_return_reliability.csv'), index=False)
    print(stage+' probability metrics (all runners, not ROI):')
    print(pd.DataFrame(metrics).query("fold == 'all'").to_string(index=False), flush=True)


def save_bet_diagnostics(details, directory, prefix):
    """Diagnostic candidate graphs remain explicitly separate from deployed bets."""
    bought = details.loc[details['buy_flag']].copy()
    hits = bought.loc[bought['hit_bought']].copy()
    hits.to_csv(directory/(prefix+'_hit_odds.csv'), index=False)
    bins = np.linspace(0, 1, 11)
    rows = []
    for label, part in bought.groupby(
            pd.cut(bought['predicted_place_probability'], bins, include_lowest=True), observed=True):
        stake, payout = float(part['stake'].sum()), float(part['payout'].sum())
        rows.append(dict(probability_bin=str(label), probability_bin_midpoint=float(label.mid), bets=len(part), hits=int(part['hit_bought'].sum()),
                         average_probability=float(part['predicted_place_probability'].mean()),
                         hit_rate=float(part['hit_bought'].mean()), stake=stake, payout=payout,
                         profit=payout-stake, roi=payout/stake if stake else np.nan))
    table = pd.DataFrame(rows, columns=['probability_bin','probability_bin_midpoint','bets','hits','average_probability',
                                       'hit_rate','stake','payout','profit','roi'])
    table.to_csv(directory/(prefix+'_probability_hits.csv'), index=False)
    quarterly = []
    for period, part in bought.groupby(pd.to_datetime(bought['race_date']).dt.to_period('Q')):
        quarterly.append(dict(quarter=str(period), **summarize_settlement(part)))
    pd.DataFrame(quarterly).to_csv(directory/(prefix+'_quarterly_results.csv'), index=False)
    fig, ax = plt.subplots(2, 2, figsize=(13, 9))
    if len(bought):
        x = table['probability_bin_midpoint'].to_numpy()
        ax[0,0].bar(x, table['bets'], width=.065, alpha=.3, label='Bought tickets')
        ax[0,0].bar(x, table['hits'], width=.04, color='#167E83', label='Hit tickets')
        ax[0,0].legend()
        ax[0,1].scatter(hits['predicted_place_probability'], hits['return_target'], alpha=.5, color='#167E83')
        day = bought.groupby('race_date')[['stake','payout','profit']].sum().sort_index()
        ax[1,0].plot(day.index, day['profit'].cumsum(), color='#167E83')
        ax[1,1].plot(day.index, day['payout'].cumsum()/day['stake'].cumsum(), color='#167E83')
    else:
        for a in ax.flat:
            a.text(.5,.5,'No bets',ha='center',transform=a.transAxes)
    ax[0,0].set(title='Predicted probability and hit ticket counts', xlabel='Predicted PLACE probability', ylabel='Tickets')
    ax[0,1].set(title='Purchased hits: realized gross payout odds', xlabel='Predicted PLACE probability', ylabel='Payout / stake')
    for a in ax[0]:
        a.set_xlim(0, 1)
        a.xaxis.set_major_formatter(PercentFormatter(1))
    ax[1,0].set(title='Cumulative flat-stake profit', ylabel='Simulated stake currency')
    ax[1,0].axhline(0, color='grey', linestyle='--')
    ax[1,1].set(title='Cumulative gross ROI', ylabel='Payout / stake')
    ax[1,1].axhline(1, color='grey', linestyle='--')
    for a in ax[1]:
        a.tick_params(axis='x', rotation=30)
    for a in ax.flat:
        a.grid(alpha=.15)
    status = 'FIXED CANDIDATE / DIAGNOSTIC' if prefix.startswith('shadow') else 'DEPLOYMENT-GATED STRATEGY'
    fig.suptitle(directory.name+' | '+status+' | exploratory historical data', fontsize=11)
    fig.tight_layout()
    fig.savefig(directory/(prefix+'_graphs.png'), dpi=150, bbox_inches='tight')
    if SHOW_PLOTS:
        plt.show()
    plt.close(fig)



def save_v12_venue_report(predictions, outcomes, policy, directory, stage):
    """Observed ROI only in explicitly labeled diagnostic datasets.

    This report NEVER feeds the selected policy. Test results are reused/exploratory.
    """
    rows = []
    for scope in ('ALL', 'ST', 'HV'):
        p = dict(policy, venue_filter=scope)
        d = settle_plan(make_bet_plan(predictions, p, active=True), outcomes)
        bought = d.loc[d['buy_flag']]
        stats = summarize_settlement(d)
        lo, mid, hi = bootstrap_roi(d)
        rows.append({'stage':stage, 'filter':scope, 'model':policy['model'],
                     'score':policy['score'], **stats,
                     'weekly_bootstrap_low95':lo, 'weekly_bootstrap_high95':hi,
                     'unique_weeks': int(pd.to_datetime(bought['race_date']).dt.to_period('W').nunique()) if len(bought) else 0,
                     'status': ('exploratory_reused_test_NOT_selection' if 'test' in stage else 'diagnostic_NOT_selection')})
    pd.DataFrame(rows).to_csv(Path(directory) / f'v12_{stage}_venue_diagnostics.csv', index=False)


def assert_v12_pre_race_policy():
    """Fast adversarial smoke test: no settlement column can affect purchases."""
    dates = pd.to_datetime(['2020-01-01']*4)
    example = pd.DataFrame({'row_id':range(4), 'race_id':['a','a','b','b'],
        'race_date':dates, 'racecourse':['ST','ST','HV','HV'], 'race_no':[1,1,1,1],
        'horse_number':[1,2,1,2], 'demo_p':[0.5,0.45,0.55,0.4],
        'demo_blend':[1.5,1.4,1.2,0.7]})
    for scope, expected in [('ALL',{0,2}), ('ST',{0}), ('HV',{2})]:
        policy={'model':'demo','score':'blend','min_return':1.1,
            'min_probability':0.3,'top_n':1,'venue_filter':scope}
        original=make_bet_plan(example, policy)
        tampered=example.assign(return_target=1000, place_hit=0, finish_position=1,
                                win_odds=1.0, actual_result=42)
        after=make_bet_plan(tampered, policy)
        assert set(original.loc[original.buy_flag,'row_id']) == expected
        assert original['buy_flag'].equals(after['buy_flag'])
    return True

def run_period(all_runners, odds, cfg, output_root):
    directory = Path(output_root) / cfg['name']
    directory.mkdir(parents=True, exist_ok=True)
    start, end = pd.Timestamp(cfg['train_start']), pd.Timestamp(cfg['train_end'])
    test_start, test_end = pd.Timestamp(cfg['test_start']), pd.Timestamp(cfg['test_end'])
    if not start < end < test_start <= test_end:
        raise ValueError('Require train_start < train_end < test_start <= test_end.')
    history_raw = all_runners.loc[all_runners['race_date'].between(start, end)].copy()
    test_raw = all_runners.loc[all_runners['race_date'].between(test_start, test_end)].copy()
    if history_raw.empty or test_raw.empty:
        raise ValueError('No rows in a configured training/test period.')
    history, audit = attach_outcomes(history_raw, odds, cfg['name'] + '_development')
    audit.to_csv(directory / 'development_audit.csv', index=False, encoding='utf-8-sig')
    dates = np.sort(history['race_date'].unique())
    cuts = [int(len(dates) * p) for p in [0.40, 0.55, 0.70, 0.85]]
    if cuts[0] < MIN_FIT_DATES + MIN_CALIBRATION_DATES or min(np.diff(cuts)) < 8:
        raise ValueError('Insufficient race dates for the fixed chronological validation design.')
    oof_parts, provenance = [], []
    for fold, (a, b) in enumerate(zip(cuts[:-1], cuts[1:]), start=1):
        stage_start = time.perf_counter()
        prior = history.loc[history['race_date'] < dates[a]]
        future = history.loc[history['race_date'].between(dates[a], dates[b - 1])]
        print(f"{cfg['name']}: chronological discovery fold {fold}/3", flush=True)
        predictions, audit_fit = fit_predict_block(prior, future, outcome_loader=lambda day: day)
        TIMING_REPORT.append({'period': cfg['name'], 'stage': f'discovery_fold_{fold}', 'seconds': time.perf_counter()-stage_start})
        predictions['fold_id'] = fold
        oof_parts.append(predictions)
        provenance.append({'stage': 'discovery', 'fold': fold, **audit_fit})
        if fold == 1:
            pd.DataFrame({'feature_name': audit_fit['features']}).to_csv(
                directory / 'features_used_train_only.csv', index=False)
            pd.DataFrame({'excluded_constant_or_all_missing':
                          audit_fit['dropped_noninformative_features']}).to_csv(
                directory / 'features_excluded_train_only.csv', index=False)
            print(f"{cfg['name']}: model uses {audit_fit['feature_count']} past-only features; "
                  f"excluded {len(audit_fit['dropped_noninformative_features'])} constant/missing", flush=True)
    oof = pd.concat(oof_parts, ignore_index=True)
    oof.to_csv(directory / 'development_oof_predictions.csv', index=False)
    save_prediction_diagnostics(oof, history, directory, 'discovery')
    development_probability_ablation(oof, history, directory)
    stage_start = time.perf_counter()
    policy, discovery_pass, search = validation_search(oof, history)
    TIMING_REPORT.append({'period': cfg['name'], 'stage': 'policy_search', 'seconds': time.perf_counter()-stage_start})
    search.to_csv(directory / 'validation_search.csv', index=False)
    write_json(directory / 'v14_control_challenger_comparison.json', search.attrs['v14_comparison'])
    save_return_decomposition(settle_plan(make_bet_plan(oof, policy), history), directory, 'discovery')
    save_v12_venue_report(oof, history, policy, directory, 'discovery')
    save_v12_discovery_diagnostics(oof, history, policy, directory)
    # Diagnostic visibility does not alter thresholds or look at the test set.
    diagnostic_columns = ['model', 'score', 'venue_filter', 'min_return', 'min_probability',
                          'top_n', 'bought_tickets', 'roi', 'fold_bet_counts',
                          'fold_rois', 'worst_fold_roi', 'selection_lower90',
                          'sample_qualified', 'support_fraction',
                          'eligible', 'rejection_reasons']
    search[diagnostic_columns].head(12).to_csv(
        directory / 'top_12_discovery_policies_DIAGNOSTIC.csv', index=False)
    # All tables below are from discovery folds only. The test is not used to
    # choose a feature set, model, threshold, or fallback.
    comparison = (search.groupby(['model','score','venue_filter'], as_index=False)
                  .agg(best_discovery_roi=('roi','max'),
                       max_development_bets=('bought_tickets','max'),
                       eligible_policies=('eligible','sum')))
    comparison.to_csv(directory/'discovery_model_comparison.csv', index=False)
    print(f'{cfg["name"]}: discovery model comparison (diagnostic; max ROI is selection-biased)')
    print(comparison.to_string(index=False), flush=True)
    # Print subgroup evidence without using confirmation/test data to change policy.
    venue_summary = search.groupby('venue_filter', as_index=False).agg(
        candidate_policies=('eligible','size'),
        passing_policies=('eligible','sum'),
        best_discovery_roi=('roi','max'),
        best_bootstrap_lower90=('selection_lower90','max'),
        max_bets=('bought_tickets','max'))
    venue_summary.to_csv(directory/'v12_discovery_venue_candidate_summary.csv',index=False)
    print(f"{cfg['name']}: venue-scope discovery diagnostics (ROI maxima are selection-biased)")
    print(venue_summary.to_string(index=False), flush=True)
    print(f"{cfg['name']}: discovery eligible policies: {int(search['eligible'].sum())} / {len(search)}")
    print(search[diagnostic_columns].head(3).to_string(index=False), flush=True)
    # Diagnostic snapshots compare policy-activity buckets from discovery only.
    # No test outcomes are used in these tables or in policy selection.
    print(f"{cfg['name']}: best sample-qualified discovery policies (not necessarily deployable)", flush=True)
    supported = search.loc[search['sample_qualified']].head(5)
    if len(supported):
        print(supported[['model','score','venue_filter','min_return','min_probability','top_n',
                         'bought_tickets','roi','fold_rois','selection_lower90','eligible']].to_string(index=False))
    # One chosen strategy is checked once on the final 15% of development dates.
    confirmation_start = dates[cuts[-1]]
    prior = history.loc[history['race_date'] < confirmation_start]
    confirmation = history.loc[history['race_date'] >= confirmation_start]
    chosen_profile = (MODEL_PROFILES.copy() if policy['model'] in ('ensemble', V11_HYBRID_NAME)
                      else {policy['model']: MODEL_PROFILES[policy['model']]})
    print(f"{cfg['name']}: independent confirmation block", flush=True)
    stage_start = time.perf_counter()
    pred_confirm, audit_fit = fit_predict_block(prior, confirmation, chosen_profile, outcome_loader=lambda day: day)
    TIMING_REPORT.append({'period': cfg['name'], 'stage': 'confirmation_models', 'seconds': time.perf_counter()-stage_start})
    provenance.append({'stage': 'confirmation', **audit_fit})
    save_prediction_diagnostics(pred_confirm, confirmation, directory, 'confirmation')
    confirm_details = settle_plan(make_bet_plan(pred_confirm, policy), confirmation)
    confirm_summary = summarize_settlement(confirm_details)
    save_return_decomposition(confirm_details, directory, 'confirmation')
    save_confirmation_diagnostics(pred_confirm, confirmation, policy, confirm_details, directory)
    save_v12_venue_report(pred_confirm, confirmation, policy, directory, 'confirmation')
    print(cfg['name'], 'diagnostic policy selected from discovery:', policy,
          '(discovery eligibility:', discovery_pass, ')',
          '\nconfirmation:', confirm_summary, flush=True)
    rejection_reasons = []
    if not discovery_pass:
        rejection_reasons.append('discovery policy failed risk/sample-size gates')
    if confirm_summary['bought_tickets'] < MIN_CONFIRMATION_BETS:
        rejection_reasons.append(f"confirmation sample too small ({confirm_summary['bought_tickets']} < {MIN_CONFIRMATION_BETS})")
    if not (confirm_summary['roi'] > 1.0):
        rejection_reasons.append('confirmation ROI at or below 1.0')
    if rejection_reasons:
        print(cfg['name'], ': NO DEPLOYMENT:', '; '.join(rejection_reasons), flush=True)
    confirm_pass = bool(confirm_summary['bought_tickets'] >= MIN_CONFIRMATION_BETS
                        and confirm_summary['roi'] > 1.0)
    active = discovery_pass and confirm_pass
    confirm_details.to_csv(directory / 'confirmation_tickets.csv', index=False)
    lock = {'v14_development_comparison': search.attrs['v14_comparison'],
            'v14_value_enabled': V14_VALUE_ENABLED, 'v14_value_iterations': V14_VALUE_ITERATIONS,
            'period': cfg, 'policy': policy, 'discovery_pass': discovery_pass,
            'confirmation_pass': confirm_pass, 'active': active,
            'confirmation_results': confirm_summary,
            'rejection_reasons': rejection_reasons,
            'unit_stake': UNIT_STAKE, 'evaluation': 'flat stake, no bankroll constraint',
            'history_mode': HISTORY_MODE,
            'model_and_policy_test_refitting': False,
            'test_previously_reported_in_input_notebook': True, 'v13_history_sharing': SHARE_HISTORY_FEATURE_PASSES, 'v13_fast_policy_search': FAST_POLICY_SEARCH, 'v13_confirmation_coverage_buffer': MIN_PROJECTED_CONFIRMATION_BETS,
            'v11_fixed_blend_weights': {'legacy': V11_LEGACY_WEIGHT, 'compact': V11_COMPACT_WEIGHT},
            'v12_course_gate': policy.get('venue_filter','ALL'),
            'configuration': {'models': MODEL_PROFILES, 'iterations': BOOSTING_ITERATIONS,
                              'return_thresholds': RETURN_THRESHOLDS, 'minimum_probabilities': MIN_PROBABILITIES,
                              'top_n_options': TOP_N_OPTIONS, 'v12_scoped_venues': V12_SCOPED_VENUES, 'v12_scoped_models': V12_SCOPED_MODELS, 'min_validation_bets': MIN_VALIDATION_BETS,
                              'min_confirmation_bets': MIN_CONFIRMATION_BETS, 'min_fold_roi': MIN_FOLD_ROI},
            'current_race_odds_used': False, 'place_probability_race_projection': True, 'final_test_retuning_allowed': False,
            'prior_test_outcomes_already_viewed': True,
            'new_test_claim_valid': False}
    # Commit policy before test replay. Daily mode uses earlier completed dates only.
    write_json(directory / 'locked_strategy.json', lock)
    if not active and not RUN_SHADOW_FOR_REJECTED:
        # Locked no-bet policy means there is no test bet to predict or settle.
        # This avoids an expensive and unnecessary final refit and prevents
        # reusing already-examined test outcomes for exploratory shadow scoring.
        print(f"{cfg['name']}: locked policy; active=False; skipping unused final-test training", flush=True)
        summary = {'bought_tickets': 0, 'bought_races': 0, 'hit_tickets': 0,
                   'unresolved_bought_tickets': 0, 'total_stake': 0.0,
                   'total_payout': 0.0, 'total_profit': 0.0, 'roi': np.nan,
                   'audited_roi': np.nan, 'hit_rate': np.nan,
                   'period': cfg['name'], 'active': False,
                   'evaluation_status': 'rejected_before_test; no test model fitted',
                   'shadow_candidate_roi_not_deployed': np.nan,
                   'shadow_candidate_tickets_not_deployed': 0,
                   'shadow_bootstrap_low95': np.nan, 'shadow_bootstrap_high95': np.nan,
                   'bootstrap_low95': np.nan, 'bootstrap_high95': np.nan,
                   'observed_roi_over_1': False, 'sufficient_test_sample': False,
                   'conservative_ci_above_1': False}
        pd.DataFrame(columns=IDENTIFIERS + ['buy_flag','ticket_units']).to_csv(
            directory / 'locked_test_bet_plan.csv', index=False)
        write_json(directory / 'final_test_summary.json', summary)
        write_json(directory / 'prediction_provenance.json', {
            'stages': provenance, 'final_test_models_fitted': False,
            'reason': 'strategy failed pre-test discovery/confirmation'})
        print(pd.DataFrame([summary]).to_string(index=False), flush=True)
        return summary
    print(f"{cfg['name']}: locked policy; active={active}; predicting final test", flush=True)
    def completed_test_day(day):
        date = day['race_date'].iloc[0].strftime('%Y-%m-%d')
        daily_odds = odds.loc[odds['race_date'].eq(date)]
        return attach_outcomes(day, daily_odds, cfg['name'] + '_past_day_' + date)[0]
    stage_start = time.perf_counter()
    predictions, audit_fit = fit_predict_block(
        history, test_raw, chosen_profile, outcome_loader=completed_test_day)
    TIMING_REPORT.append({'period': cfg['name'], 'stage': 'final_test_models', 'seconds': time.perf_counter()-stage_start})
    provenance.append({'stage': 'final_test', **audit_fit})
    plan = make_bet_plan(predictions, policy, active=active)
    # Shadow uses the same policy and chronological prediction snapshots.
    # It is an evaluation-only diagnostic when evidence gates reject deployment.
    shadow_plan = make_bet_plan(predictions, policy, active=True)
    plan.to_csv(directory / 'locked_test_bet_plan.csv', index=False)
    shadow_plan.to_csv(directory / 'locked_shadow_test_bet_plan.csv', index=False)
    digest = hashlib.sha256((directory / 'locked_test_bet_plan.csv').read_bytes()).hexdigest()
    write_json(directory / 'prediction_provenance.json', {'stages': provenance, 'test_plan_sha256': digest})
    # Settlement report: in daily mode earlier-date outcomes were used only as history.
    # No alternative policy is tried after seeing test ROI.
    test_outcomes, test_audit = attach_outcomes(test_raw, odds, cfg['name'] + '_final_test')
    test_audit.to_csv(directory / 'test_audit.csv', index=False, encoding='utf-8-sig')
    details = settle_plan(plan, test_outcomes)
    details.to_csv(directory / 'final_test_tickets.csv', index=False)
    details.loc[details['hit_bought']].to_csv(directory / 'final_test_hit_odds.csv', index=False)
    summary = summarize_settlement(details)
    shadow_details = settle_plan(shadow_plan, test_outcomes)
    shadow_summary = summarize_settlement(shadow_details)
    shadow_details.to_csv(directory / 'shadow_test_tickets_DIAGNOSTIC.csv', index=False)
    save_v12_venue_report(predictions, test_outcomes, policy, directory, 'shadow_test')
    save_bet_diagnostics(shadow_details, directory, 'shadow_test')
    save_bet_diagnostics(details, directory, 'deployed_test')
    save_return_decomposition(details, directory, 'deployed_test')
    low, _, high = bootstrap_roi(details)
    shadow_low, _, shadow_high = bootstrap_roi(shadow_details)
    summary.update({'period': cfg['name'], 'active': active,
                    'evaluation_status': 'exploratory_reused_test_not_independent',
                    'shadow_candidate_roi_not_deployed': shadow_summary['roi'],
                    'shadow_candidate_tickets_not_deployed': shadow_summary['bought_tickets'],
                    'shadow_bootstrap_low95': shadow_low,
                    'shadow_bootstrap_high95': shadow_high,
                    'bootstrap_low95': low,
                    'bootstrap_high95': high,
                    'observed_roi_over_1': bool(summary['bought_tickets'] > 0 and summary['roi'] > 1.0),
                    'sufficient_test_sample': bool(summary['bought_tickets'] >= MIN_FINAL_TEST_BETS),
                    'conservative_ci_above_1': bool(np.isfinite(low) and low > 1.0)})
    write_json(directory / 'final_test_summary.json', summary)
    plot_final_report(details, cfg['name'], directory)
    print(pd.DataFrame([summary]).to_string(index=False), flush=True)
    return summary


# Dataset autodiscovery. Keep the four-space indents below when copying.
def resolve_input(configured, basename):
    explicit = Path(configured)
    if explicit.is_file():
        return explicit
    roots = [Path('/kaggle/input'), Path.cwd(), Path.home() / 'Downloads']
    for root in roots:
        candidate = root / basename
        if candidate.is_file():
            return candidate
        if root == Path('/kaggle/input') and root.exists():
            matches = list(root.rglob(basename))
            if len(matches) == 1:
                return matches[0]
    raise FileNotFoundError(f'Required dataset missing: {basename}. Set its path in the configuration cell.')


def main():
    data_path = resolve_input(DATA_PATH, 'train_2014_to_2022.xlsx')
    odds_path = resolve_input(ODDS_PATH, 'hkjc_Complete_all_dividends_20140914_20220930.csv')
    Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
    print('Loading:', data_path, odds_path, flush=True)
    # Cache only *preparation*, not model fits, test results, or selected policy.
    # Key includes source absolute path, file length and modification nanoseconds.
    source_fingerprint = hashlib.sha256(
        f'{data_path.resolve()}|{data_path.stat().st_size}|{data_path.stat().st_mtime_ns}|v13-prep1'.encode()
    ).hexdigest()[:16]
    cache_path = Path(OUTPUT_DIR) / f'runners_{source_fingerprint}.pkl'
    if CACHE_PARSED_RUNNERS and cache_path.is_file():
        runners = pd.read_pickle(cache_path)
        print('Loaded prepared runners from local cache:', cache_path, flush=True)
    else:
        t_read = time.perf_counter()
        raw = pd.read_excel(data_path)
        save_dataset_audit(raw, Path(OUTPUT_DIR))
        runners = prepare_runners(raw)
        if CACHE_PARSED_RUNNERS:
            tmp_path = cache_path.with_suffix('.pkl.tmp')
            runners.to_pickle(tmp_path)
            tmp_path.replace(cache_path)
        print(f'Excel import + runner preparation: {time.perf_counter()-t_read:.1f}s', flush=True)
    odds = normalize_odds_for_eval(pd.read_csv(odds_path))
    print('HISTORY_MODE:', HISTORY_MODE, 'SHARE_HISTORY_FEATURE_PASSES:', SHARE_HISTORY_FEATURE_PASSES,
          'RUN_SHADOW_FOR_REJECTED:', RUN_SHADOW_FOR_REJECTED, flush=True)
    print('NOTICE: These historical test periods have ALREADY been inspected. V14 results are exploratory,')
    print('not new independent out-of-sample evidence. Verify later on genuinely unseen dates.')
    summaries = [run_period(runners, odds, cfg, OUTPUT_DIR) for cfg in TRAIN_TEST_PERIODS]
    table = pd.DataFrame(summaries)
    table.to_csv(Path(OUTPUT_DIR) / 'roi_summary.csv', index=False)
    # Aggregate money across disjoint test periods; never average per-period ROI.
    intervals = sorted((pd.Timestamp(c['test_start']), pd.Timestamp(c['test_end'])) for c in TRAIN_TEST_PERIODS)
    if any(b[0] <= a[1] for a, b in zip(intervals, intervals[1:])):
        print('Combined totals omitted: overlapping test periods would double count bets.')
    else:
        total_stake = float(table['total_stake'].sum())
        total_payout = float(table['total_payout'].sum())
        combined = {'total_stake': total_stake, 'total_payout': total_payout,
                    'total_profit': total_payout-total_stake,
                    'roi': total_payout/total_stake if total_stake else np.nan,
                    'bought_tickets': int(table['bought_tickets'].sum()),
                    'hit_tickets': int(table['hit_tickets'].sum()),
                    'evaluation_status': 'exploratory_reused_historical_tests'}
        write_json(Path(OUTPUT_DIR)/'combined_deployed_summary.json', combined)
        print('Combined deployed results:', combined, flush=True)
    times = pd.DataFrame(TIMING_REPORT)
    times.to_csv(Path(OUTPUT_DIR) / 'v14_runtime_breakdown.csv', index=False)
    print('Execution breakdown (seconds):')
    print(times.to_string(index=False), flush=True)
    display(table)
    return table


if __name__ == '__main__':
    assert assert_v12_pre_race_policy(), 'V12 pre-race policy leakage smoke test failed'
    roi_summary = main()

# V12 notes: V9/V10 controls and V11 hybrid are unchanged; limited course gates compete during discovery.
# Current-race odds are NOT available as usable pre-bet prices, so real ROI may not improve.
# The full dividend file is required for actual profitability evaluation.

Loading: /kaggle/input/datasets/miwawa/train-2014-to-2022/train_2014_to_2022.xlsx /kaggle/input/datasets/miwawa/all-odds-1/hkjc_Complete_all_dividends_20140914_20220930.csv
Dataset audit: {'rows': 81695, 'columns': 55, 'last_call_equals_finish_fraction': 0.9993628743113421, 'current_odds_used': False, 'current_sectional_times_used': False, 'source_and_split_columns_used': False, 'note': 'The result-file track condition/weights are assumed available at bet time; verify operational snapshots.'}
Excel import + runner preparation: 58.2s
HISTORY_MODE: daily SHARE_HISTORY_FEATURE_PASSES: True RUN_SHADOW_FOR_REJECTED: False
NOTICE: These historical test periods have ALREADY been inspected. V14 results are exploratory,
not new independent out-of-sample evidence. Verify later on genuinely unseen dates.

実着順とPLACE配当馬番が一致しないレース:
 race_date racecourse  race_no place_odds_audit_reason actual_paid_horses odds_paid_horses actual_only_horses odds_only_horses
2015-01-28         HV        6 paid horse s

,bought_tickets,bought_races,hit_tickets,unresolved_bought_tickets,total_stake,total_payout,total_profit,roi,audited_roi,hit_rate,...,evaluation_status,shadow_candidate_roi_not_deployed,shadow_candidate_tickets_not_deployed,shadow_bootstrap_low95,shadow_bootstrap_high95,bootstrap_low95,bootstrap_high95,observed_roi_over_1,sufficient_test_sample,conservative_ci_above_1
0,222,222,71,0,111000.0,126400.0,15400.0,1.138739,1.138739,0.31982,...,exploratory_reused_test_not_independent,1.138739,222,0.826785,1.525183,0.826785,1.525183,True,True,False
1,0,0,0,0,0.0,0.0,0.0,NaN,NaN,NaN,...,rejected_before_test; no test model fitted,NaN,0,NaN,NaN,NaN,NaN,False,False,False
